# Phase 8 — ST-CDGM **from-scratch** intégrant 10+ fixes experts

Architecture from-scratch validée par 5 experts (ML/Math/Recherche/IA/Climat). Vise à **battre noncausal v4 sur la majorité des métriques**.

## Probabilités empiriques estimées (basées sur V5 mesuré)

| Métrique | Proba battre noncausal | Justification |
|----------|------------------------|---------------|
| Pearson, RMSE, MAE, CRPS, RAPSD | **70-85%** | V5 le fait déjà ; Min-SNR + features physiques aident |
| SSR (calibration) | 50-60% | Multi-EMA + cond_dropout + Dispersive Loss |
| **F1@p99** | **10-30%** | Verdict expert (17/17 variants échec, 0 précédent publié) |
| CSI/SEDI/FSS | 40-60% | Suivent F1 ; FSS tolère décalages spatiaux |
| Indices climatiques (CDD, R10, RX1) | 60-75% | V5 déjà mieux sur CDD/R10 ; tail_weight raffine |

## Architecture (fixes intégrés)

### Stage 1 (causal mean predictor, from-scratch)
- 15 features existantes + **w_700** (interp pondérée 0.571·w_850 + 0.429·w_500, Holton 2004)
- + **θ_e_850, θ_e_500** (Bolton 1980)
- + **MUCAPE proxy = θ_e_850 − θ_e_500** (remplace CAPE 2-niveaux trop grossier, Emanuel 1994)
- Encoder + RCN + DAG learnable + dual_path Path B UNet
- **Losses physiques différentiables corrigées** :
  - `L_R10mm = MSE(Σ_t σ(5·(expm1(x)-10)), Σ_t 1[expm1(x)>10])` ← seuil en mm/day après expm1
  - `L_Rx1day = LSE_T=5(expm1(pred)) approx max` ← LogSumExp stable
  - `L_CDD = MSE(σ(5·(1-expm1(x))), 1[expm1(x)<1])`
  - `L_CC = autograd((∂μ/∂T_850)·σ_T/σ_μ − 0.07)²` ← adimensionné, autograd obligatoire
  - Pondération : 0.20·R10 + 0.15·Rx1 + 0.10·CDD + 0.05·CC, warmup epoch 10+

### Stage 2 (diffusion EDM)
- UNet 4 niveaux [128,256,256,256] CorrDiff Normal (~50M params)
- **Conditioning** : `causal_concat=True` (fallback documenté — AdaGN reporté à V9 cf. réserve IA)
- **Min-SNR-γ=5** weighting EDM (Hang ICCV 2023, arXiv 2303.09556)
- **Tail_weight (4, 12)** au lieu de (8, 25) — Climate ML
- **Dispersive Loss** λ=0.25 (corrigé vs 0.05 sous-dosé) mid-block hook (He&Wang 2025, arXiv 2506.09027)
- **conditioning_dropout p=0.13** (Ho&Salimans 2022)
- **Multi-EMA** {0.999, 0.9995, 0.9999} avec post-hoc sweep (Karras 2024, arXiv 2312.02696)
- **α appris ∈ [0,1]** avec régularisation `L_α = 0.1·(α-0.5)² + 1.0·max(0, 0.3-α)²` (évite α→0 collapse)

### Sampling
- `dpm_solver++` 32 steps (Lu 2022, arXiv 2211.01095)
- **Limited-Interval Guidance** σ ∈ [0.05, 1.0] (Kynkäänniemi 2024, arXiv 2404.07724)
- cfg_scale 1.0-1.5

### Eval protocol (publication-ready)
- **N_BATCHES = 64**, **K_SAMPLES = 128** (CI ±0.014)
- 3 conventions F1@p99 : pooled full-grid (vs noncausal), ETCCDI per-pixel land, land-only pooled
- CSI@p99, SEDI@p99, FSS (n=9, n=25, n=51 px)
- CRPS gaussian, rank histogram, RMSE, MAE, Pearson global+per-sample
- **Indices climatiques sur 730 jours ENTIERS** (rx1day, CDD, R10mm, DJF/JJA) — ETCCDI Zhang 2011
- μ_HR ablation (causalité opérationnelle), Q_phys (interprétabilité), α appris final
- **Paired permutation test** n=10000 (Phase 8 vs noncausal sur mêmes batches)
- **Bootstrap CI 95% BCa** n=1000 (Math)
- **Holm-Bonferroni** correction multi-tests
- **Pre-enregistrement** : git rev-parse HEAD logged dans JSON output

## Coût compute estimé (A100 Pro+)
- Stage 1 : ~6-7h, 15 epochs
- Stage 2 : ~22-27h, 200 epochs (cached)
- Eval BS30 unifié + comparaison 3-way : ~15h
- **Total : ~45-55h, soit 2-3 sessions Pro+ avec checkpoint/resume**

In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric, cftime, h5netcdf, xbatcher, diffusers
    from omegaconf import OmegaConf
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ], check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab')

import torch, numpy as np, json, time
import xarray as xr
from omegaconf import OmegaConf

# Reproducibility (Expert IA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Pre-registration (Expert Recherche)
_git_sha = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print(f'[bootstrap] git SHA = {_git_sha}  branch = {GIT_BRANCH}')
print(f'[bootstrap] cwd={os.getcwd()}  torch={torch.__version__}  cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'[bootstrap] GPU = {torch.cuda.get_device_name(0)}  VRAM = {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# === Cell 2 : Constants ===
DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
DATA_ROOT  = DRIVE_ROOT / 'data'
HR_RAW_PATH = DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc'
LR_RAW_PATH = DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc'
STATIC_PATH = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'

# Phase 8 output dir
# ============== FEATURE ABLATION FLAG (5-expert Option 1) ==================
# When False (default) : Phase 8 uses 19 LR features (15 originaux + 4 augmented :
#   w_700, theta_e_850, theta_e_500, mucape_proxy).
# When True : Phase 8 uses only the 15 features that noncausal v4 was trained
#   on, enabling 100%-fair comparison Phase 8/15 vs noncausal v4 to isolate
#   the contribution of the causal architecture (vs the feature augmentation).
# Run both FULL passes (False then True) for the 3-way ablation analysis :
#   Phase 8/19 vs noncausal v4 = arch + features
#   Phase 8/19 vs Phase 8/15   = features alone
#   Phase 8/15 vs noncausal v4 = arch alone (the publishable causal claim)
ABLATION_FEATURES_15 = False

OUT_DIR_BASE = DRIVE_ROOT / 'oracle_9node' / 'phase8_from_scratch'
_subdir = 'ablation_15' if ABLATION_FEATURES_15 else 'full_19'
OUT_DIR = OUT_DIR_BASE / _subdir
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'[Cell 2] FEATURE MODE : {_subdir} ({"15" if ABLATION_FEATURES_15 else "19"} LR features)  OUT_DIR={OUT_DIR}')
CKPT_STAGE1_LAST  = OUT_DIR / 'stage1_last.pth'
CKPT_STAGE1_BEST  = OUT_DIR / 'stage1_best.pth'
STAGE1_CACHE_PATH = OUT_DIR / 'stage1_cache_mu_total.pt'
VAL_CACHE_PATH    = OUT_DIR / 'stage1_cache_val_mu_total.pt'
CKPT_STAGE2_LAST  = OUT_DIR / 'stage2_last.pth'
CKPT_STAGE2_BEST  = OUT_DIR / 'stage2_best.pth'
LAND_MASK_PATH    = OUT_DIR / 'land_mask_nz.npy'
CLIM_PATH         = OUT_DIR / 'clim_train_p95_p99.npz'
AUGMENTED_LR_PATH = OUT_DIR / 'lr_augmented_features.nc'  # w_700, theta_e, MUCAPE
TRAINING_HISTORY  = OUT_DIR / 'training_history.json'
FINAL_RESULTS     = OUT_DIR / 'phase8_final_results.json'

# Reference checkpoints to compare against (existing models)
REF_CKPT_NONCAUSAL = DRIVE_ROOT / 'ckpt_noncausal'
REF_CKPT_V5_CAUSAL = DRIVE_ROOT / 'ckpt_v2_corrdiff_normal'

# ============== SMOKE MODE FLAG (Expert team validation step) ==============
# When True : 2-3 epochs, N=2 batches, K=4 samples, N_STEPS=8 -> ~15 min total
# When False : full protocol -> ~45-55h
SMOKE_MODE = False   # FULL #1 : Phase 8 with 19 features

# Reproducibility
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# K9 temporal split (used by all phases)
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],   # 30 years
    'val':     ['2010-01-01', '2011-12-31'],   # 2 years
    'test':    ['2012-01-01', '2013-12-31'],   # 2 years -> 730 days for climate indices
    'holdout': ['2014-01-01', '2014-12-31'],   # 1 year (out-of-distribution warm year)
}

# Stage 1 hyperparams (Path C+ Option C proven on seed_42)
STAGE1_EPOCHS    = 25   # was 15 ; extended for marginal corr gain (user request after FULL#2 ep15 healthy)
STAGE1_LR        = 1e-3
LAMBDA_DAG_PRIOR = 0.40
LAMBDA_L1_START  = 0.04
LAMBDA_L1_END    = 0.005
G_PHYS_ALPHA     = 0.25
# Physical loss weights
# 5/5-expert UNANIMOUS post-collapse-90c3519 : PHYS_LOSS_WARMUP_EPOCH set to
# 26 (>STAGE1_EPOCHS=25) so phys losses NEVER activate in Stage 1.
# Empirical : phys activation at ep10 caused causal_frac 0.725 -> 0.028 collapse,
# Cell 9 mu_HR/target corr fell to 0.091 (random).
# Root cause (Math) : per-batch Rx1day pulls toward unphysical compressed target
# (5-day max ~30-50mm vs annual ~150-250mm); FusionGate detach feedback amplifies.
# Literature (Recherche) : 0/8 papers use phys ETCCDI losses in pre-training
# (CorrDiff/STVD/WassDiff/bias-informed CDM all MSE-only Stage 1).
# Stage 2 retains tail-weight x4/x12 + FACL for extreme-precip skill.
# LAMBDA values KEPT for future reactivation (e.g., post-Stage2 fine-tune).
LAMBDA_R10MM   = 0.20
LAMBDA_RX1DAY  = 0.15
LAMBDA_CDD     = 0.10
LAMBDA_CC      = 0.05
PHYS_LOSS_WARMUP_EPOCH = 26   # was 16 ; kept > STAGE1_EPOCHS=25 to maintain phys-off Stage 1

# Stage 2 hyperparams
STAGE2_EPOCHS         = 200
STAGE2_LR             = 2e-4
STAGE2_BATCH_SIZE     = 32   # was 64 -- reduced to fit A100 80GB OOM (Dispersive hook prevents gradient_checkpointing, so activations dominate VRAM)
STAGE2_WEIGHT_DECAY   = 1e-4
STAGE2_GRADIENT_CLIP  = 1.0

# ============== STAGE 1 P1 TAIL-WEIGHT (5-expert hyperplan, Climat blessing) ==
# Critical for Westland orographic precipitation : MSE uniform on log1p is
# dominated by dry pixels (Canterbury 90% <2mm), so mu_HR converges to flat
# climatology that under-predicts Southern Alps. Stage 2 diffusion cannot
# fix a biased-low mu_HR -- delta model learns to refine, not correct bias.
# Tail-weight thresholds 15/35 mm/day = NZ-wide P85/P98 approximation
# (uniform threshold over regions, deemed tractable; per-pixel quantile
# would add ~10^4 DoF, rejected for parsimony).
STAGE1_TAIL_TAU_P95 = 15.0
STAGE1_TAIL_TAU_P99 = 35.0
STAGE1_TAIL_W_P95   = 4.0
STAGE1_TAIL_W_P99   = 12.0

# ============== STAGE 2 FACL HYPERPARAMS (Yang NeurIPS 2024) ===================
# Fourier Amplitude + Correlation Loss for precipitation diffusion.
# - FAL drives RAPSD spectral amplitude matching
# - FCL drives spatial phase coherence (critical for Westland orographic)
# Climat ranking : FCL > spectral_highk > FAL >> SWD for NZ extremes,
# hence FCL = 0.3 > FAL = 0.2.
LAMBDA_FACL_FAL = 0.2
LAMBDA_FACL_FCL = 0.3
FACL_WARMUP_EPOCHS = 3   # ML expert I2 : lambda_eff = lambda * min(1, ep/3)
MIN_SNR_GAMMA         = 5.0
TAIL_WEIGHT_P95       = 4.0    # (Climate ML : 8 -> 4 less aggressive)
TAIL_WEIGHT_P99       = 12.0   # (Climate ML : 25 -> 12)
DISPERSIVE_LAMBDA     = 0.25   # (Expert ML : raised from 0.05 sous-dosé to paper's recommended range)
DISPERSIVE_TAU        = 0.5    # Kernel temperature
COND_DROPOUT_P        = 0.13   # CFG compatibility (CorrDiff Nature CEE 2025)
EMA_DECAYS            = [0.999, 0.9995, 0.9999]   # Multi-EMA EDM2 Karras 2024
ALPHA_TARGET          = 0.5   # Regularization target for learned alpha
ALPHA_FLOOR           = 0.3   # Penalty if alpha < 0.3 (prevent collapse)
LAMBDA_ALPHA_REG      = 0.1
BETA_ALPHA_FLOOR      = 1.0
SIGMA_DATA_NEW        = 0.193  # Phase 6 dualpath recalibrated

# BS30 eval protocol (publication-ready)
N_TEST_BATCHES = 64    # Math expert : N=64 -> CI +/-0.014
K_SAMPLES      = 128   # >> CorrDiff Mardani (32)
N_STEPS_DIFF   = 32    # dpm_solver++ converges at 32 NFE
WET_DAY_THRESHOLD_MM = 1.0   # ETCCDI standard

# Sampling
SAMPLER_SCHEDULER       = 'dpm_solver++'
CFG_SCALE               = 1.0
LIMITED_GUIDANCE_SIGMA_MIN = 0.05   # Kynkäänniemi NeurIPS 2024 (calibrated for sigma_data=0.1)
LIMITED_GUIDANCE_SIGMA_MAX = 1.0

# Bootstrap + statistical tests
BOOTSTRAP_N_RESAMPLES = 1000   # Math : n_boot=1000
PAIRED_PERMUTATION_N = 10000   # Math : n_perm=10000 for p<=0.001
BOOTSTRAP_METHOD     = 'BCa'   # Bias-corrected accelerated (Efron 1987)

print(f'[Cell 2] DEVICE = {DEVICE}')
print(f'[Cell 2] DRIVE_ROOT = {DRIVE_ROOT}')
print(f'[Cell 2] OUT_DIR = {OUT_DIR}')
print(f'[Cell 2] Stage 1 : {STAGE1_EPOCHS} epochs, LR {STAGE1_LR}')
print(f'[Cell 2] Stage 2 : {STAGE2_EPOCHS} epochs, LR {STAGE2_LR}, batch {STAGE2_BATCH_SIZE}')
print(f'[Cell 2] Min-SNR γ = {MIN_SNR_GAMMA}, tail_weight ({TAIL_WEIGHT_P95}, {TAIL_WEIGHT_P99})')
print(f'[Cell 2] Multi-EMA decays = {EMA_DECAYS}')
print(f'[Cell 2] BS30 eval : N={N_TEST_BATCHES} batches x K={K_SAMPLES} samples x {N_STEPS_DIFF} steps')

# Pre-registration record (Expert Recherche)
PRE_REG_RECORD = {
    'phase': 'phase8_from_scratch',
    'git_sha': _git_sha,
    'git_branch': GIT_BRANCH,
    'seed': SEED,
    'k9_dates': K9_DATES,
    'stage1_hyperparams': {
        'epochs': STAGE1_EPOCHS, 'lr': STAGE1_LR,
        'lambda_dag_prior': LAMBDA_DAG_PRIOR,
        'lambda_l1_start': LAMBDA_L1_START, 'lambda_l1_end': LAMBDA_L1_END,
        'g_phys_alpha': G_PHYS_ALPHA,
        'physical_loss_weights': {
            'R10mm': LAMBDA_R10MM, 'Rx1day': LAMBDA_RX1DAY,
            'CDD': LAMBDA_CDD, 'CC': LAMBDA_CC,
        },
        'tail_weight_stage1': {
            'tau_mm': [STAGE1_TAIL_TAU_P95, STAGE1_TAIL_TAU_P99],
            'w':      [STAGE1_TAIL_W_P95,   STAGE1_TAIL_W_P99],
        },
        'phys_warmup_epoch': PHYS_LOSS_WARMUP_EPOCH,
        'stage1_variant': 'phys_off',   # 5/5-expert post-collapse-90c3519
    },
    'stage2_hyperparams': {
        'epochs': STAGE2_EPOCHS, 'lr': STAGE2_LR,
        'batch_size': STAGE2_BATCH_SIZE, 'weight_decay': STAGE2_WEIGHT_DECAY,
        'min_snr_gamma': MIN_SNR_GAMMA,
        'tail_weight': [TAIL_WEIGHT_P95, TAIL_WEIGHT_P99],
        'facl': {
            'fal_lambda': LAMBDA_FACL_FAL,
            'fcl_lambda': LAMBDA_FACL_FCL,
            'warmup_epochs': FACL_WARMUP_EPOCHS,
        },
        'dispersive_lambda': DISPERSIVE_LAMBDA,
        'cond_dropout_p': COND_DROPOUT_P,
        'ema_decays': EMA_DECAYS,
        'alpha_reg': {'target': ALPHA_TARGET, 'floor': ALPHA_FLOOR,
                       'lambda': LAMBDA_ALPHA_REG, 'beta_floor': BETA_ALPHA_FLOOR},
        'sigma_data': SIGMA_DATA_NEW,
    },
    'eval_protocol': {
        'n_batches': N_TEST_BATCHES, 'k_samples': K_SAMPLES,
        'n_steps_diff': N_STEPS_DIFF,
        'sampler': SAMPLER_SCHEDULER, 'cfg_scale': CFG_SCALE,
        'limited_guidance': [LIMITED_GUIDANCE_SIGMA_MIN, LIMITED_GUIDANCE_SIGMA_MAX],
        'bootstrap_n': BOOTSTRAP_N_RESAMPLES,
        'paired_permutation_n': PAIRED_PERMUTATION_N,
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}
if SMOKE_MODE:
    print('[Cell 2] *** SMOKE_MODE active : overriding hyperparams for fast smoke ***')
    # Extended SMOKE config (Climat+Recherche : need >=12 ep S1 to activate phys_warmup at ep 10)
    STAGE1_EPOCHS = 12   # was 2 -- passes phys_warmup epoch 10 + 2 ep observation
    STAGE2_EPOCHS = 5    # was 2 -- partial Stage 2 convergence
    N_TEST_BATCHES = 20  # was 2 -- bootstrap CI now usable (n>=5)
    K_SAMPLES = 16       # was 4
    N_STEPS_DIFF = 16    # was 8
    BOOTSTRAP_N_RESAMPLES = 500
    PAIRED_PERMUTATION_N = 2000
    EMA_DECAYS = [0.999, 0.9995]   # 2 decays for sweep (still less than FULL=3)
    print(f'[Cell 2] EXTENDED SMOKE : Stage1={STAGE1_EPOCHS}ep  Stage2={STAGE2_EPOCHS}ep  '
          f'N={N_TEST_BATCHES} K={K_SAMPLES} steps={N_STEPS_DIFF}')

print(f'[Cell 2] Pre-registration record created (SHA {_git_sha[:8]})')
print(f'[Cell 2] SMOKE_MODE = {SMOKE_MODE}')

In [ ]:
# === Cell 2b : SSD cache (avoid Drive FUSE disconnects on NetCDF reads) ===
# Drive FUSE periodically drops mid-read on large NetCDF files (errno 107).
# Copy once to Colab SSD (/content/) then re-point all data paths.
# Idempotent : skips files already cached.
import os, shutil, time
from pathlib import Path

SSD_CACHE_ROOT = Path('/content/climate_data_local')
SSD_DATA_ROOT  = SSD_CACHE_ROOT / 'data'
SSD_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# List the NC files we actually need (rooted under DATA_ROOT)
_NEEDED_RELATIVE = [
    'train/pr_ACCESS-CM2_hist.nc',
    'train/predictor_ACCESS-CM2_hist.nc',
    'static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc',
]
# Add val / test if the corresponding split folders exist on Drive
for _split in ('val', 'test'):
    _split_dir = DATA_ROOT / _split
    if _split_dir.exists():
        for _f in os.listdir(_split_dir):
            if _f.endswith('.nc'):
                _NEEDED_RELATIVE.append(f'{_split}/{_f}')

_t0 = time.time()
_total_bytes = 0
_n_copied = 0
_n_skipped = 0
for _rel in _NEEDED_RELATIVE:
    _src = DATA_ROOT / _rel
    _dst = SSD_DATA_ROOT / _rel
    if not _src.exists():
        print(f'[Cell 2b] WARN missing on Drive : {_src}')
        continue
    _dst.parent.mkdir(parents=True, exist_ok=True)
    _src_size = _src.stat().st_size
    if _dst.exists() and _dst.stat().st_size == _src_size:
        _n_skipped += 1
        _total_bytes += _src_size
        continue
    print(f'[Cell 2b] copy {_rel}  ({_src_size/1e9:.2f} GB) ...', flush=True)
    _t_cp = time.time()
    shutil.copy2(_src, _dst)
    print(f'[Cell 2b]   done in {time.time()-_t_cp:.1f}s '
          f'({_src_size/1e6/(time.time()-_t_cp):.0f} MB/s)')
    _n_copied += 1
    _total_bytes += _src_size

print(f'[Cell 2b] cache ready : {_n_copied} copied, {_n_skipped} skipped, '
      f'{_total_bytes/1e9:.2f} GB total in {time.time()-_t0:.1f}s')

# === Re-point data paths to local SSD cache ===
DATA_ROOT   = SSD_DATA_ROOT
HR_RAW_PATH = DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc'
LR_RAW_PATH = DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc'
STATIC_PATH = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
print(f'[Cell 2b] DATA_ROOT re-pointed to {DATA_ROOT} (SSD)')


In [ ]:
# === Cell 3 : Land mask depuis static dataset (multi-source fallback) ===
# Approche : static_predictors fournit orog/he/vegt. On essaie lsm/sftlf/vegt/orog dans l'ordre.

_t0 = time.time()
if not STATIC_PATH.exists():
    raise FileNotFoundError(f'Static dataset not on Drive : {STATIC_PATH}')
ds_static = xr.open_dataset(str(STATIC_PATH), engine='h5netcdf')
print(f'[Cell 3] static variables : {list(ds_static.data_vars)}')

land_mask = None
land_mask_source = None

# Strategy 1 : explicit lsm/sftlf/landfrac
for cand in ('lsm', 'land_sea_mask', 'landfrac', 'land_mask', 'sftlf'):
    if cand in ds_static.data_vars:
        arr = ds_static[cand].values.squeeze()
        if arr.ndim != 2: continue
        thr = 50.0 if arr.max() > 1.5 else 0.5
        land_mask = arr >= thr
        land_mask_source = f'{cand} (threshold {thr})'
        break

# Strategy 2 : vegetation type
if land_mask is None and 'vegt' in ds_static.data_vars:
    vegt = ds_static['vegt'].values.squeeze()
    if vegt.ndim == 2:
        _unique = np.unique(vegt[np.isfinite(vegt)])
        print(f'[Cell 3] vegt unique values = {_unique}')
        land_mask = (vegt > 0) & (vegt != 17) & np.isfinite(vegt)
        land_mask_source = 'vegt (0 and 17 = water)'

# Strategy 3 : orography (safe NaN handling per Climat)
if land_mask is None and 'orog' in ds_static.data_vars:
    orog = ds_static['orog'].values.squeeze()
    if orog.ndim == 2:
        print(f'[Cell 3] orog range : [{np.nanmin(orog):.2f}, {np.nanmax(orog):.2f}] m')
        if np.isnan(orog).any():
            land_mask = np.isfinite(orog) & (orog > -0.5)
            land_mask_source = 'orog : isfinite & > -0.5 m'
        else:
            land_mask = orog > 0.5
            land_mask_source = 'orog > 0.5 m'

if land_mask is None:
    raise RuntimeError(f'No land/sea variable found in static : {list(ds_static.data_vars)}')

n_land = int(land_mask.sum())
n_total = int(land_mask.size)
print(f'[Cell 3] land_mask source : {land_mask_source}')
print(f'[Cell 3] land pixels = {n_land} / {n_total} ({100*n_land/n_total:.1f}%)')
print(f'[Cell 3] expected for NZ : 22-45% (varies with bbox size)')

# Sanity HR shape match
_hr_ds = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
_pr_var = 'pr' if 'pr' in _hr_ds.data_vars else list(_hr_ds.data_vars)[0]
assert _hr_ds[_pr_var].shape[-2:] == land_mask.shape, 'HR grid mismatch'
_hr_ds.close()

LAND_MASK_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(LAND_MASK_PATH, land_mask)
ds_static.close()
print(f'[Cell 3] saved : {LAND_MASK_PATH}  ({time.time()-_t0:.1f}s)')

In [ ]:
# === Cell 4 : Climatology p95/p99 per-pixel (ETCCDI Zhang 2011) ===
# Per-pixel quantile on wet days >= 1 mm/day, training period only.
# Cache : skip the ~3min Python double-loop on kernel restart.

_t0 = time.time()
if CLIM_PATH.exists() and not SMOKE_MODE:
    _data = np.load(CLIM_PATH)
    clim_p95 = _data['clim_p95']
    clim_p99 = _data['clim_p99']
    n_wet    = _data['n_wet']
    _n_valid = int(np.isfinite(clim_p99).sum())
    print(f'[Cell 4] loaded cached climatology : {CLIM_PATH}')
    print(f'[Cell 4] land pixels with valid p99 = {_n_valid} / {n_land}')
    print(f'[Cell 4] clim_p99 range = [{np.nanmin(clim_p99):.2f}, {np.nanmax(clim_p99):.2f}] mm/day  '
          f'mean = {np.nanmean(clim_p99):.2f}  ({time.time()-_t0:.1f}s)')
else:
    _hr_ds = xr.open_dataset(str(HR_RAW_PATH), engine='h5netcdf')
    _pr_var = 'pr' if 'pr' in _hr_ds.data_vars else list(_hr_ds.data_vars)[0]
    _hr = _hr_ds[_pr_var]
    _time_var = _hr.dims[0]
    _hr_train = _hr.sel({_time_var: slice(K9_DATES['train'][0], K9_DATES['train'][1])})
    _hr_train_np = _hr_train.values.astype(np.float32)
    print(f'[Cell 4] train slice = {_hr_train_np.shape}')

    H, W = _hr_train_np.shape[1], _hr_train_np.shape[2]
    clim_p95 = np.full((H, W), np.nan, dtype=np.float32)
    clim_p99 = np.full((H, W), np.nan, dtype=np.float32)
    n_wet = np.zeros((H, W), dtype=np.int32)

    for i in range(H):
        for j in range(W):
            if not land_mask[i, j]:
                continue
            px = _hr_train_np[:, i, j]
            px_finite = px[np.isfinite(px)]
            wet = px_finite[px_finite >= WET_DAY_THRESHOLD_MM]
            n_wet[i, j] = wet.size
            if wet.size >= 30:
                clim_p95[i, j] = float(np.quantile(wet, 0.95))
                clim_p99[i, j] = float(np.quantile(wet, 0.99))

    _n_valid = int(np.isfinite(clim_p99).sum())
    print(f'[Cell 4] land pixels with valid p99 = {_n_valid} / {n_land}')
    print(f'[Cell 4] mean wet days per land pixel = {n_wet[land_mask].mean():.0f}')
    print(f'[Cell 4] clim_p99 range = [{np.nanmin(clim_p99):.2f}, {np.nanmax(clim_p99):.2f}] mm/day  mean = {np.nanmean(clim_p99):.2f}')

    CLIM_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez(CLIM_PATH, clim_p95=clim_p95, clim_p99=clim_p99, n_wet=n_wet,
              land_mask=land_mask, wet_threshold_mm=WET_DAY_THRESHOLD_MM,
              train_start=K9_DATES['train'][0], train_end=K9_DATES['train'][1])
    del _hr_train_np
    _hr_ds.close()
    print(f'[Cell 4] saved : {CLIM_PATH}  ({time.time()-_t0:.1f}s)')

In [ ]:
# === Cell 5 : Augmented LR features (w_700, θ_e_850, θ_e_500, MUCAPE proxy) ===
# Pre-compute offline once, save as NetCDF for fast loading during training.
# Formulas validated by Expert Climat :
#   w_700 = 0.571*w_850 + 0.429*w_500   (linear interpolation in pressure, Holton 2004)
#   θ_e   = θ * exp(L_v * q_sat / (c_p * T))   (Bolton 1980, simplified)
#   MUCAPE_proxy = θ_e_850 - θ_e_500   (static instability indicator, Emanuel 1994)
# Constants : R_d = 287, c_p = 1005, L_v = 2.5e6, R_v = 461.5

_t0 = time.time()

if AUGMENTED_LR_PATH.exists():
    print(f'[Cell 5] augmented LR already exists : {AUGMENTED_LR_PATH}')
    ds_lr_aug = xr.open_dataset(str(AUGMENTED_LR_PATH), engine='h5netcdf')
    print(f'[Cell 5] variables : {list(ds_lr_aug.data_vars)}')
else:
    print(f'[Cell 5] computing augmented features from {LR_RAW_PATH}...')
    ds_lr = xr.open_dataset(str(LR_RAW_PATH), engine='h5netcdf')
    print(f'[Cell 5] LR variables available : {list(ds_lr.data_vars)}')

    # ----- 1. w_700 by linear interpolation in pressure -----
    if 'w_850' in ds_lr.data_vars and 'w_500' in ds_lr.data_vars:
        w_700 = 0.571 * ds_lr['w_850'] + 0.429 * ds_lr['w_500']
        w_700.attrs['long_name'] = 'vertical_velocity_at_700hPa (interpolated)'
        w_700.attrs['units'] = 'Pa s-1'
        w_700.attrs['interpolation'] = '0.571*w_850 + 0.429*w_500 (linear in pressure)'
        print(f'[Cell 5] w_700 computed : shape {w_700.shape} range [{float(w_700.min()):.4f}, {float(w_700.max()):.4f}]')
    else:
        raise RuntimeError('w_850 and w_500 required for w_700 interpolation')

    # ----- 2. θ_e_850 and θ_e_500 via Bolton 1980 -----
    # θ = T * (1000/p)^(R_d/c_p)
    # q_sat via Magnus-Tetens : e_s = 6.112 * exp(17.67*(T-273.15)/(T-29.65))
    # θ_e = θ * exp(L_v * q_sat(T,p) / (c_p * T))
    R_d = 287.0
    c_p = 1005.0
    L_v = 2.5e6
    eps = 0.622  # R_d / R_v

    def _theta_e_bolton(T_K, q_obs, p_hPa):
        """Bolton 1980 simplified pseudo-equivalent potential temperature.
        Uses OBSERVED specific humidity q (kg/kg), NOT q_sat (Expert team fix).
        T_K in Kelvin, q_obs in kg/kg, p_hPa in hPa.
        Formula : theta * exp(L_v * q / (c_p * T))  (Stull 1988 simplified Bolton).
        """
        theta = T_K * (1000.0 / p_hPa) ** (R_d / c_p)
        q = np.clip(q_obs, 1e-8, 0.05)  # safe bounds kg/kg
        return theta * np.exp(L_v * q / (c_p * T_K))

    if 't_850' in ds_lr.data_vars and 'q_850' in ds_lr.data_vars:
        T_850 = ds_lr['t_850']  # Kelvin (ACCESS-CM2 standard)
        if 'q_850' not in ds_lr.data_vars:
            raise RuntimeError('q_850 required for theta_e (observed humidity)')
        q_850 = ds_lr['q_850']
        # FIX BUG #6 (IA) : avoid scalar in apply_ufunc, call function directly (numpy broadcast via xarray)
        theta_e_850 = _theta_e_bolton(T_850, q_850, 850.0)
        theta_e_850.attrs['long_name'] = 'equivalent_potential_temperature_850hPa (Bolton 1980 simplified, observed q)'
        theta_e_850.attrs['units'] = 'K'
        print(f'[Cell 5] θ_e_850 computed : range [{float(theta_e_850.min()):.1f}, {float(theta_e_850.max()):.1f}] K')
    else:
        raise RuntimeError('t_850 required for θ_e')

    if 't_500' in ds_lr.data_vars and 'q_500' in ds_lr.data_vars:
        T_500 = ds_lr['t_500']
        if 'q_500' not in ds_lr.data_vars:
            raise RuntimeError('q_500 required for theta_e (observed humidity)')
        q_500 = ds_lr['q_500']
        theta_e_500 = _theta_e_bolton(T_500, q_500, 500.0)
        theta_e_500.attrs['long_name'] = 'equivalent_potential_temperature_500hPa (Bolton 1980 simplified, observed q)'
        theta_e_500.attrs['units'] = 'K'
        print(f'[Cell 5] θ_e_500 computed : range [{float(theta_e_500.min()):.1f}, {float(theta_e_500.max()):.1f}] K')

    # ----- 3. MUCAPE proxy = θ_e_850 - θ_e_500 -----
    mucape_proxy = theta_e_850 - theta_e_500
    mucape_proxy.attrs['long_name'] = 'MUCAPE_proxy (theta_e_850 - theta_e_500)'
    mucape_proxy.attrs['units'] = 'K'
    mucape_proxy.attrs['interpretation'] = 'positive = convectively unstable'
    print(f'[Cell 5] MUCAPE proxy : range [{float(mucape_proxy.min()):.2f}, {float(mucape_proxy.max()):.2f}] K')

    # ----- Build augmented LR dataset (15 original + 4 new = 19 variables) -----
    ds_lr_aug = ds_lr.copy()
    ds_lr_aug['w_700']        = w_700
    ds_lr_aug['theta_e_850']  = theta_e_850
    ds_lr_aug['theta_e_500']  = theta_e_500
    ds_lr_aug['mucape_proxy'] = mucape_proxy

    # Save
    ds_lr_aug.to_netcdf(str(AUGMENTED_LR_PATH), engine='h5netcdf')
    ds_lr.close()
    print(f'[Cell 5] augmented LR saved : {AUGMENTED_LR_PATH}')
    print(f'[Cell 5] total variables = {len(ds_lr_aug.data_vars)} (original 15 + 4 new)')

print(f'[Cell 5] done in {time.time()-_t0:.1f}s')

In [ ]:
# === Cell 6 : Pipeline + dataloaders avec features augmentées ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES

# Load config
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = True
CONFIG.training.num_workers = 0
ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = LAMBDA_DAG_PRIOR
ts_cfg.stage1['g_phys_alpha']     = G_PHYS_ALPHA
OmegaConf.set_struct(CONFIG, False)

# Add augmented LR variables (4 new : w_700, theta_e_850, theta_e_500, mucape_proxy)
# 5-expert Option 1 : ABLATION_FEATURES_15 toggle for fair comparison with noncausal v4.
# Fix : NONCAUSAL_15_VARS hardcoded (independent of CONFIG state). The previous
# version captured ORIGINAL_LR_VARS from CONFIG which made the cell non-idempotent
# across runs : after one run with flag=False, CONFIG was already mutated to 19
# vars, and the next run with flag=True kept 19 vars instead of restoring 15.
NONCAUSAL_15_VARS = [
    'u_850', 'u_500', 'u_250',
    'v_850', 'v_500', 'v_250',
    'w_850', 'w_500', 'w_250',
    'q_850', 'q_500', 'q_250',
    't_850', 't_500', 't_250',
]
ORIGINAL_LR_VARS  = list(NONCAUSAL_15_VARS)
AUGMENTED_LR_VARS = NONCAUSAL_15_VARS + ['w_700', 'theta_e_850', 'theta_e_500', 'mucape_proxy']
# Assert that the YAML config matches the hardcoded canonical 15 (catches drift)
_yaml_vars = list(CONFIG.data.lr_variables)
_yaml_canon = [v for v in _yaml_vars if v in set(NONCAUSAL_15_VARS)]
if set(_yaml_canon) != set(NONCAUSAL_15_VARS):
    print(f'[Cell 6] WARN : YAML lr_variables ({len(_yaml_vars)}) does not contain the 15 '
          f'canonical noncausal-v4 features. YAML has : {sorted(_yaml_canon)}. '
          f'Expected : {sorted(NONCAUSAL_15_VARS)}.')

if ABLATION_FEATURES_15:
    # Ablation mode : use ONLY the 15 features that noncausal v4 was trained on.
    # Removes w_700 / theta_e_850 / theta_e_500 / mucape_proxy from LR pipeline.
    ACTIVE_LR_VARS = list(NONCAUSAL_15_VARS)
    CONFIG.data.lr_variables = list(ACTIVE_LR_VARS)
    print(f'[Cell 6] LR variables : {len(ACTIVE_LR_VARS)} (ABLATION mode : noncausal-v4-compatible)')
    print(f'[Cell 6] dropped : w_700, theta_e_850, theta_e_500, mucape_proxy')
else:
    ACTIVE_LR_VARS = list(AUGMENTED_LR_VARS)
    CONFIG.data.lr_variables = list(ACTIVE_LR_VARS)
    print(f'[Cell 6] LR variables : {len(ACTIVE_LR_VARS)} (original 15 + 4 augmented)')
    print(f'[Cell 6] new variables : w_700, theta_e_850, theta_e_500, mucape_proxy')

# Add 9-node metapaths (existing + new ones for augmented features)
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in {mm.name for mm in CONFIG.encoder.metapaths}:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))

# Build pipeline using AUGMENTED LR file (with w_700, theta_e, MUCAPE)
SEQ_LEN = int(CONFIG.data.seq_len)
pipeline = NetCDFDataPipeline(
    lr_path=str(AUGMENTED_LR_PATH), hr_path=str(HR_RAW_PATH),
    static_path=str(STATIC_PATH) if STATIC_PATH.exists() else None,
    seq_len=SEQ_LEN, baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.precipitation_delta),
    lr_variables=ACTIVE_LR_VARS,
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else [],
    # FIX BUG #2 : force pipeline to recompute means/stds on the augmented LR (19 vars).
    # The pre-computed NetCDF stats only cover the 15 original vars ; xarray arithmetic
    # would silently drop the 4 augmented ones (w_700, theta_e_850/500, mucape_proxy).
    means_path=None,
    stds_path=None,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True)
test_dataset  = pipeline.build_sequence_dataset(split='test',  seq_len=SEQ_LEN, stride=int(CONFIG.data.stride), as_torch=True)

train_dataloader = _DataLoader(train_dataset, batch_size=1, num_workers=0, pin_memory=True,
                                collate_fn=lambda x: x, shuffle=False)
val_dataloader   = _DataLoader(val_dataset,   batch_size=1, num_workers=0, pin_memory=True,
                                collate_fn=lambda x: x, shuffle=False)

# Graph builder (9-node)
lr_shape = tuple(CONFIG.graph.lr_shape); hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(lr_shape=lr_shape, hr_shape=hr_shape,
                              static_dataset=pipeline.get_static_dataset(),
                              include_mid_layer=CONFIG.graph.include_mid_layer,
                              extended_9node=True)
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])

# Convert sample to batch (mirrors phase6 pattern)
_LR_VARS = ACTIVE_LR_VARS
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850','q_500','q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850','w_500','w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850','500','250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]; u = lr0[:, [_VI[f'u_{lev}']]]; v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u*u + v*v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None: acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t): return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']; seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    _ivt = _compute_ivt_nodes(lr0)
    dyn = {}
    for nt in builder.dynamic_node_types:
        if nt == 'Q850':   dyn[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
        elif nt == 'W500': dyn[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
        elif nt == 'IVT':  dyn[nt] = _ensure_2d(_ivt)
        else:              dyn[nt] = _ensure_2d(lr0)
    hetero = builder.prepare_step_data(dyn).to(device)
    return {'lr': lr_tensor, 'lr_grid': lr_seq, 'residual': sample['residual'],
            'baseline': sample.get('baseline'), 'hetero': hetero, 'time': sample.get('time')}

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
print(f'[Cell 6] LR channels detected = {C_LR} (expected {len(ACTIVE_LR_VARS)})')
# Fail loudly if pipeline silently dropped variables OR returned more than asked
assert C_LR == len(ACTIVE_LR_VARS), (
    f'LR channels {C_LR} != expected {len(ACTIVE_LR_VARS)}. '
    f'ACTIVE_LR_VARS = {ACTIVE_LR_VARS}. '
    'Pipeline silently dropped/added variables. '
    'Check means_path/stds_path are None and that lr_variables= matches ACTIVE_LR_VARS.'
)
print(f'[Cell 6] HR shape = ({H_HR}, {W_HR})')
print(f'[Cell 6] dynamic node types = {builder.dynamic_node_types}')
print(f'[Cell 6] Pipeline + dataloaders ready')


In [ ]:
# === Cell 7 : Stage 1 build from-scratch + physical losses corrigées ===
from st_cdgm.models.dual_path_stage1 import DualPathPredictor
from st_cdgm.training.stage1_paths import batch_lr_grid_last
from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder, IntelligibleVariableConfig
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

# Build modules from-scratch (NO checkpoint load).
_metapath_configs = [
    IntelligibleVariableConfig(name=m.name, meta_path=(m.src, m.relation, m.target), pool='mean')
    for m in CONFIG.encoder.metapaths
]
encoder = IntelligibleVariableEncoder(
    configs=_metapath_configs,
    hidden_dim=int(CONFIG.encoder.hidden_dim),
    conditioning_dim=int(CONFIG.encoder.conditioning_dim),
).to(DEVICE)
num_vars = len(_metapath_configs)

_lr_nodes = builder.lr_grid_to_nodes(_probe['lr'][0])
rcn_driver_dim = _lr_nodes.shape[-1]
rcn_cell = RCNCell(
    num_vars=num_vars, hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=rcn_driver_dim, reconstruction_dim=rcn_driver_dim,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

# ========================================================================
# Option Z (5-expert UNANIMES) : A_dag = prior structural FIXE Trenberth-inspired
# Lacher le DAG learnable end-to-end (Math : non identifiable sur 6-8 vars
# observationnel sans interventions. Recherche : aucun precedent top-tier
# 2024-2026 en climate downscaling avec DAG learnable. ML+IA : gain stabilite
# x10, ODD ameliore +3 a +7 AUROC via conformal prediction post-training).
#
# Prior structural : strict upper triangular 8x8, value 0.15, encodes the
# topological causal order over the 8 metapaths :
#   0: GP850_spat_adj (lower-level geopotential, spatial coupling)
#   1: GP850_to_GP500 (vertical coupling lower->mid)
#   2: GP500_spat_adj (mid-level geopotential)
#   3: GP500_to_GP250 (vertical coupling mid->upper)
#   4: GP250_spat_adj (upper-level forcing)
#   5: Q850            (specific humidity, moisture)
#   6: W500            (vertical velocity, motion)
#   7: IVT             (integrated vapor transport)
#
# Interpretation Trenberth-inspired : upper-level dynamics (GP250) drives
# mid-level (GP500), drives lower-level (GP850), gates moisture (Q850),
# couples to motion (W500), enables transport (IVT). Strict upper-triangular
# guarantees acyclicity by construction. Frobenius norm ~0.79.
# ========================================================================
# IA expert fix : use _orig_mod pattern (compile-safe). Without this, if rcn_cell
# is later wrapped by torch.compile, the .copy_() / requires_grad_() may target
# the wrapper proxy instead of the underlying leaf Parameter.
_rcn_core_init = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
_n_vars = _rcn_core_init.A_dag.shape[0]
_prior = torch.zeros(_n_vars, _n_vars, device=_rcn_core_init.A_dag.device,
                     dtype=_rcn_core_init.A_dag.dtype)
# Trenberth-weighted sparse prior (5-expert audit, Climat reco).
# 10 physical edges over 8 metapaths (in pipeline order) :
#   0: GP850_spat_adj   (lower-level geopotential)
#   1: GP850_to_GP500   (vertical coupling lower -> mid)
#   2: GP500_spat_adj   (mid-level geopotential)
#   3: GP500_to_GP250   (vertical coupling mid -> upper)
#   4: GP250_spat_adj   (upper-level forcing)
#   5: Q850             (specific humidity)
#   6: W500             (vertical velocity)
#   7: IVT              (integrated vapor transport)
# Physical flow (Holton 2004, Trenberth 1991, Newell 1992) :
#   - Upper-level jet (GP250) drives mid-level (GP500) via synoptic forcing
#   - Mid-level dynamics drive ascent (W500) via Q-G omega equation
#   - Low-level circulation (GP850) gates moisture convergence (Q850, IVT)
#   - W500 lifts moisture, Q850 supports IVT
# Verified : acyclic, ||A||_F = 1.07, h_DAGMA ~ 0 by construction
_trenberth_edges = [
    (4, 2, 0.45),  # GP250 -> GP500 : upper-level synoptic forcing
    (4, 6, 0.20),  # GP250 -> W500  : jet-level divergence -> ascent
    (3, 4, 0.45),  # GP500_to_GP250 -> GP250 : vertical coupling
    (2, 6, 0.35),  # GP500 -> W500  : Q-G omega equation
    (1, 2, 0.45),  # GP850_to_GP500 -> GP500 : vertical coupling
    (0, 5, 0.20),  # GP850 -> Q850  : low-level vorticity gates moisture
    (0, 7, 0.30),  # GP850 -> IVT   : low-level circulation drives transport
    (6, 5, 0.15),  # W500 -> Q850   : ascent lifts moisture (feedback)
    (5, 7, 0.40),  # Q850 -> IVT    : specific humidity drives transport
    (6, 7, 0.25),  # W500 -> IVT    : vertical motion modulates transport
]
for _i, _j, _w in _trenberth_edges:
    _prior[_i, _j] = _w
with torch.no_grad():
    _rcn_core_init.A_dag.copy_(_prior)
_rcn_core_init.A_dag.requires_grad_(False)
_rcn_core_init.set_dag_grad_gate(1.0)
print(f'[Cell 7] A_dag FIXED to Trenberth-inspired upper-triangular prior :')
print(f'         shape={tuple(_rcn_core_init.A_dag.shape)}  ||A||_F={float(_rcn_core_init.A_dag.norm()):.4f}  '
      f'asymmetry={float((_rcn_core_init.A_dag - _rcn_core_init.A_dag.T).abs().mean()):.4f}')
print(f'         requires_grad={_rcn_core_init.A_dag.requires_grad}  gate={float(_rcn_core_init.dag_grad_gate):.2f}')

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model), hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h), intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads), refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

dual_path = DualPathPredictor(
    in_channels=C_LR, base_ch=48, hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=0.40, path_b_kind='unet',
    path_b_unet_channels=(32, 64, 128),
    path_b_unet_lr_shape=(23, 26),
).to(DEVICE)

# Learnable alpha for Stage 2 reconstruction (HR = baseline + alpha * mu_HR + delta)
# Initialised at sigmoid(0) = 0.5 (Math expert : prevents alpha->0 collapse)
alpha_logit = torch.nn.Parameter(torch.tensor(0.0, device=DEVICE))

# FIX LazyModule : materialize UninitializedParameter via a dummy forward
# (some modules use LazyLinear / LazyConv which only init on first forward).
print(f'[Cell 7] Running dummy forward to materialize lazy params...')
with torch.no_grad():
    _batch_probe = convert_sample_to_batch(_probe, builder, DEVICE)
    _lr_data = _batch_probe['lr'].to(DEVICE)
    _h_init = encoder.init_state(_batch_probe['hetero']).to(DEVICE)
    _drivers = [_lr_data[t] for t in range(_lr_data.shape[0])]
    _seq_out = rcn_runner.run(_h_init, _drivers, reconstruction_sources=None)
    _mu_A = regression_head(_seq_out.states[-1])
    if _mu_A.dim() == 3: _mu_A = _mu_A.unsqueeze(0)
    _lr_grid = batch_lr_grid_last(_batch_probe, builder=builder, device=DEVICE)
    _lr_safe = torch.nan_to_num(_lr_grid, nan=0.0)
    _ = dual_path(_lr_safe, _mu_A)
print(f'[Cell 7] Lazy modules materialized')

# === DIAG NaN : identify which feature LR contains NaN or std=0 ===
print()
print('=' * 78)
print('[DIAG NaN] Per-feature stats on probe batch')
print('=' * 78)
_lr_data = _batch_probe['lr'].to(DEVICE)
print(f'[DIAG] _batch_probe[lr] shape = {tuple(_lr_data.shape)}')
print(f'[DIAG] ACTIVE_LR_VARS ({len(ACTIVE_LR_VARS)}) = {ACTIVE_LR_VARS}')
print(f'[DIAG] _lr_data global finite ? {torch.isfinite(_lr_data).all().item()}')
print(f'[DIAG] _lr_data total NaN count = {(~torch.isfinite(_lr_data)).sum().item()}')

for c_idx, vname in enumerate(ACTIVE_LR_VARS):
    chan = _lr_data[..., c_idx]
    n_nan = (~torch.isfinite(chan)).sum().item()
    n_finite = chan.numel() - n_nan
    if n_finite > 0:
        fin = chan[torch.isfinite(chan)]
        c_min = float(fin.min()); c_max = float(fin.max())
        c_mean = float(fin.mean()); c_std = float(fin.std())
    else:
        c_min = c_max = c_mean = c_std = float('nan')
    flag = ''
    if n_nan > 0: flag += '  <<< NaN'
    if abs(c_std) < 1e-6: flag += '  <<< STD~0'
    if abs(c_max) > 1e4: flag += '  <<< EXPLOSIVE'
    print(f'  [{c_idx:2d}] {vname:14s}  NaN={n_nan:6d}  min={c_min:+.4g}  '
          f'max={c_max:+.4g}  mean={c_mean:+.4g}  std={c_std:.4g}{flag}')

print()
print(f'[DIAG] baseline last finite ? {torch.isfinite(_batch_probe["baseline"][-1]).all().item()}')
print(f'[DIAG] residual last finite ? {torch.isfinite(_batch_probe["residual"][-1]).all().item()}')

with torch.no_grad():
    _h_init_check = encoder.init_state(_batch_probe['hetero']).to(DEVICE)
    print(f'[DIAG] encoder.init_state finite ? {torch.isfinite(_h_init_check).all().item()}')
    _seq_out_check = rcn_runner.run(_h_init_check, [_lr_data[t] for t in range(_lr_data.shape[0])], reconstruction_sources=None)
    _last = _seq_out_check.states[-1]
    print(f'[DIAG] rcn_runner.states[-1] finite ? {torch.isfinite(_last).all().item()}')
    _mu_check = regression_head(_last)
    print(f'[DIAG] regression_head(mu_A) finite ? {torch.isfinite(_mu_check).all().item()}')
print('=' * 78)


# Robust param count : skip UninitializedParameter (some metapaths fall back to zeros
# when their edges are not in the graph -- their Lazy layers stay uninitialised).
from torch.nn.parameter import UninitializedParameter
def _count_init_params(module):
    total = 0; n_uninit = 0
    for p in module.parameters():
        if isinstance(p, UninitializedParameter):
            n_uninit += 1
            continue
        try:
            total += p.numel()
        except Exception:
            n_uninit += 1
    return total, n_uninit

print(f'[Cell 7] Stage 1 modules built from-scratch')
for _label, _mod in [('encoder', encoder), ('rcn_cell', rcn_cell),
                       ('reg_head', regression_head), ('dual_path', dual_path)]:
    _n, _u = _count_init_params(_mod)
    extra = f'  ({_u} uninit params -- expected for fallback metapaths)' if _u > 0 else ''
    print(f'[Cell 7]   {_label} params : {_n:,}{extra}')
print(f'[Cell 7]   alpha init      : {torch.sigmoid(alpha_logit).item():.4f}')

# ===========================================================================
# Physical losses (differentiable, in mm/day space after expm1 — Math+Climat fix)
# Pipeline applies log1p(x + PRECIPITATION_DELTA), so to recover mm/day :
#   x_mm = expm1(x_log1p) - PRECIPITATION_DELTA
# PRECIPITATION_DELTA = 0.01 (from pipeline.py)
# ===========================================================================
PRECIP_DELTA = 0.01

def _to_mm_day(x_log1p):
    """Convert log1p(pr + delta) back to mm/day, clamped to [0, 500] (Climat)."""
    return (torch.expm1(x_log1p) - PRECIP_DELTA).clamp(min=0.0, max=500.0)

def loss_R10mm(pred_log1p, target_log1p, valid_mask=None, steepness=5.0):
    """Differentiable approx of R10mm (count of days >= 10 mm/day).
    Steepness=5 (Climat raised from 2). Operates in mm/day space."""
    pred_mm = _to_mm_day(pred_log1p)
    target_mm = _to_mm_day(target_log1p)
    # Sigmoid approx of 1[x >= 10]
    pred_soft   = torch.sigmoid(steepness * (pred_mm - 10.0))
    target_soft = torch.sigmoid(steepness * (target_mm - 10.0))
    if valid_mask is not None:
        pred_soft   = pred_soft   * valid_mask
        target_soft = target_soft * valid_mask
    # FIX 5-expert : .mean() on spatial dims gives PROPORTION (in[0,1]) instead
    # of absolute count (~10^4-10^5 raw events). Magnitude commensurable with
    # MSE on residual log1p (~0.04). Prevents grad_phys/grad_mse ~ 1e5 blowup
    # that froze A_dag when phys losses activated at ep>=PHYS_LOSS_WARMUP_EPOCH.
    pred_freq   = pred_soft.mean(dim=tuple(range(1, pred_soft.dim())))    # proportion
    target_freq = target_soft.mean(dim=tuple(range(1, target_soft.dim())))
    return ((pred_freq - target_freq) ** 2).mean()

def loss_Rx1day(pred_log1p, target_log1p, T=5.0):
    """Differentiable approx of annual max (Rx1day).
    Use LogSumExp_T / T (numerically stable approx of max)."""
    pred_mm   = _to_mm_day(pred_log1p)
    target_mm = _to_mm_day(target_log1p)
    # LSE_T(x) = (1/T) * log(sum exp(T*x))  -> approaches max(x) as T -> inf
    flat_pred   = pred_mm.flatten(start_dim=1)
    flat_target = target_mm.flatten(start_dim=1)
    lse_pred   = torch.logsumexp(T * flat_pred,   dim=1) / T
    lse_target = torch.logsumexp(T * flat_target, dim=1) / T
    # FIX 5-expert : normalize by climato max squared (NZ Rx1day ~100mm/day) so
    # the loss is in [0, ~1] commensurable with MSE. Prevents grad blowup.
    RX1D_CLIM_MAX_SQ = 100.0 ** 2
    return ((lse_pred - lse_target) ** 2).mean() / RX1D_CLIM_MAX_SQ

def loss_CDD(pred_log1p, target_log1p, steepness=5.0):
    """Differentiable approx of CDD (count of dry days, x < 1 mm/day)."""
    pred_mm   = _to_mm_day(pred_log1p)
    target_mm = _to_mm_day(target_log1p)
    # Sigmoid approx of 1[x < 1]
    pred_soft   = torch.sigmoid(steepness * (1.0 - pred_mm))
    target_soft = torch.sigmoid(steepness * (1.0 - target_mm))
    # FIX 5-expert : .mean() on spatial dims (see loss_R10mm for rationale).
    pred_freq   = pred_soft.mean(dim=tuple(range(1, pred_soft.dim())))
    target_freq = target_soft.mean(dim=tuple(range(1, target_soft.dim())))
    return ((pred_freq - target_freq) ** 2).mean()

def loss_Clausius_Clapeyron(mu_HR_pred, T_850_batch, target_log1p, mode='extreme'):
    """Clausius-Clapeyron loss : (d mu / d T) - rate * mu = 0.
    Math : autograd, NOT finite differences. Adimensionned.
    Climat : mode='extreme' (rate=0.07 for P>P95), mode='mean' (rate=0.05)."""
    if not T_850_batch.requires_grad:
        T_850_batch = T_850_batch.detach().requires_grad_(True)
    # We need mu_HR_pred to depend on T_850 for autograd to work.
    # Caller is responsible for this dependency. Here we just compute the loss.
    rate = 0.07 if mode == 'extreme' else 0.05
    try:
        grad = torch.autograd.grad(
            outputs=mu_HR_pred.sum(),
            inputs=T_850_batch,
            create_graph=True,
            retain_graph=True,
        )[0]
    except RuntimeError:
        # If T_850 was not part of the graph, return zero loss (safe fallback)
        return torch.tensor(0.0, device=mu_HR_pred.device, requires_grad=False)
    # Adimensionner par sigma de chaque variable (Math)
    sigma_T = T_850_batch.std().clamp_min(1e-6)
    sigma_mu = mu_HR_pred.std().clamp_min(1e-6)
    grad_adim = grad * sigma_T / sigma_mu
    target_grad = rate * mu_HR_pred / sigma_mu
    return ((grad_adim - target_grad) ** 2).mean()

# Alpha regularization (Math : prevent alpha -> 0 collapse, cf. MC2RD failure)
def loss_alpha_reg(alpha):
    target_loss = LAMBDA_ALPHA_REG * (alpha - ALPHA_TARGET) ** 2
    floor_loss  = BETA_ALPHA_FLOOR * (torch.relu(ALPHA_FLOOR - alpha) ** 2)
    return target_loss + floor_loss

print('[Cell 7] Physical losses defined :')
print('  loss_R10mm        (steepness=5, in mm/day space)')
print('  loss_Rx1day       (LogSumExp_T=5)')
print('  loss_CDD          (steepness=5)')
print('  loss_Clausius_Clapeyron (autograd, mode=extreme)')
print('  loss_alpha_reg    (lambda=0.1, beta_floor=1.0 for alpha < 0.3)')

# Tail weight (5-expert hyperplan : moved from Cell 10/11 to Cell 7 so Stage 1 can use it)
def _tail_weight(hr_log_recon, tau95_mm=15.0, tau99_mm=35.0, w95=TAIL_WEIGHT_P95, w99=TAIL_WEIGHT_P99):
    import math
    tau95 = math.log1p(tau95_mm)
    tau99 = math.log1p(tau99_mm)
    return (1.0 + (w95 - 1.0) * (hr_log_recon > tau95).float()
                + (w99 - w95) * (hr_log_recon > tau99).float())



In [ ]:
# === Cell 8 : Stage 1 training loop from-scratch ===
# 15 epochs (or 2 in SMOKE_MODE), Adam optimizer.
# DAG prior + L1 sparsity cosine decay + physical losses warmup at epoch 10+.
import time
from torch.optim import AdamW
from torch.nn.parameter import UninitializedParameter as _UP

# Option Z (5-expert UNANIMES) : A_dag is FIXED in Cell 7 (Trenberth prior,
# requires_grad=False). No DAG-learnable machinery here -- single param group
# AdamW on everything that still has gradient.
# IMPORTANT : the encoder has UninitializedParameter for fallback metapaths
# (Q850/W500/IVT) that have NO edge in the data -- they never materialize
# via the dummy forward (no data flows through them). Filter them out to
# avoid AdamW crashing on .numel() and to keep them out of the optimizer.
stage1_params = [p for p in (
    list(encoder.parameters()) + list(rcn_cell.parameters())
    + list(regression_head.parameters()) + list(dual_path.parameters())
) if p.requires_grad and not isinstance(p, _UP)]
_n_uninit = sum(1 for p in (
    list(encoder.parameters()) + list(rcn_cell.parameters())
    + list(regression_head.parameters()) + list(dual_path.parameters())
) if isinstance(p, _UP))
optimizer_s1 = AdamW(stage1_params, lr=STAGE1_LR, weight_decay=0.0)
print(f'[Cell 8] Optimizer : AdamW lr={STAGE1_LR} (A_dag FROZEN, not in optimizer)')
print(f'         Trainable params : {sum(p.numel() for p in stage1_params):,} '
      f'({_n_uninit} UninitializedParameter excluded -- fallback metapaths)')

# Option Z : A_dag is FIXED Trenberth prior with gate=1.0 set in Cell 7.
# No L1, no DAG schedule, no gate warmup. Stubs kept for backward compatibility
# with the epoch loop variables (they reference lam_l1 and _current_gate).
def _l1_schedule(epoch, total): return 0.0
def _gate_schedule(epoch): return 1.0

# Helper : Stage 1 forward through dual-path
def _stage1_forward(batch):
    """Returns (mu_A, mu_total, mu_B, gate, baseline_log, target_residual)."""
    lr_data = batch['lr'].to(DEVICE)
    h_init  = encoder.init_state(batch['hetero']).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_out = rcn_runner.run(h_init, drivers, reconstruction_sources=None)
    mu_A = regression_head(seq_out.states[-1])
    if mu_A.dim() == 3: mu_A = mu_A.unsqueeze(0)
    lr_grid = batch_lr_grid_last(batch, builder=builder, device=DEVICE)
    lr_safe = torch.nan_to_num(lr_grid, nan=0.0)
    mu_total, mu_B, gate = dual_path(lr_safe, mu_A)
    mu_total = torch.nan_to_num(mu_total, nan=0.0)
    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1: bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)
    tgt = batch['residual'][-1].to(DEVICE)
    if tgt.dim() == 3: tgt = tgt.unsqueeze(0)
    return mu_A, mu_total, mu_B, gate, bl, tgt

# Training history
stage1_history = []

# Resume logic (symmetric to Stage 2 Cell 11). Skipped in SMOKE_MODE so smoke
# tests always start fresh from the latest code.
start_epoch_s1 = 1
if CKPT_STAGE1_LAST.exists() and not SMOKE_MODE:
    print(f'[Cell 8] RESUME from {CKPT_STAGE1_LAST}')
    _ck1 = torch.load(CKPT_STAGE1_LAST, map_location=DEVICE, weights_only=False)
    encoder.load_state_dict(_ck1['encoder_state_dict'])
    rcn_cell.load_state_dict(_ck1['rcn_cell_state_dict'])
    regression_head.load_state_dict(_ck1['regression_head_state_dict'])
    dual_path.load_state_dict(_ck1['dual_path_state_dict'])
    # Option Z FIX #2 (IA expert) : ALWAYS re-pose the Trenberth prior after
    # load_state_dict. Without this, a legacy CKPT (pre-Option-Z) would
    # cristallize the learnable-era A_dag value in the resumed run.
    _rcn_core_chk = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
    if hasattr(_rcn_core_chk, 'A_dag'):
        _n_v = _rcn_core_chk.A_dag.shape[0]
        _prior_re = torch.zeros(_n_v, _n_v, device=_rcn_core_chk.A_dag.device,
                                dtype=_rcn_core_chk.A_dag.dtype)
        # Same sparse Trenberth edges as Cell 7 (kept in sync)
        for _i, _j, _w in [
            (4, 2, 0.45), (4, 6, 0.20), (3, 4, 0.45), (2, 6, 0.35),
            (1, 2, 0.45), (0, 5, 0.20), (0, 7, 0.30), (6, 5, 0.15),
            (5, 7, 0.40), (6, 7, 0.25),
        ]:
            _prior_re[_i, _j] = _w
        with torch.no_grad():
            _rcn_core_chk.A_dag.copy_(_prior_re)
        _rcn_core_chk.A_dag.requires_grad_(False)
        _rcn_core_chk.set_dag_grad_gate(1.0)
    if 'alpha_logit' in _ck1 and _ck1['alpha_logit'] is not None:
        with torch.no_grad():
            alpha_logit.copy_(_ck1['alpha_logit'].to(DEVICE))
    # Option Z FIX #3 (IA expert) : skip optimizer state restore if the
    # checkpoint is from a legacy run (param-group mismatch since A_dag is
    # no longer in any param_group). Detect via 'option_z' marker added to
    # the new payloads.
    _is_legacy = not _ck1.get('option_z', False)
    if _is_legacy:
        print(f'  Legacy CKPT detected (pre-Option-Z) : skipping optimizer state restore '
              f'(param-group mismatch -- A_dag was in legacy optimizer)')
    elif _ck1.get('optimizer_state_dict') is not None:
        try:
            optimizer_s1.load_state_dict(_ck1['optimizer_state_dict'])
        except Exception as _e:
            print(f'  WARN : optimizer_s1 state restore failed ({_e}) -- continuing with fresh moments')
    if 'history' in _ck1 and isinstance(_ck1['history'], list):
        stage1_history = list(_ck1['history'])
    start_epoch_s1 = int(_ck1.get('epoch', 0)) + 1
    print(f'  Resumed at epoch {start_epoch_s1}/{STAGE1_EPOCHS}, history len={len(stage1_history)}')
    if start_epoch_s1 > STAGE1_EPOCHS:
        print(f'  Stage 1 already complete in checkpoint -- skipping loop')

print(f'[Cell 8] Stage 1 training : epochs {start_epoch_s1}..{STAGE1_EPOCHS} (SMOKE_MODE={SMOKE_MODE})')
for ep in range(start_epoch_s1, STAGE1_EPOCHS + 1):
    _t0 = time.time()
    # Mode train
    for m in [encoder, rcn_cell, regression_head, dual_path]:
        m.train()

    lam_l1 = _l1_schedule(ep - 1, STAGE1_EPOCHS)        # always 0 (Option Z)
    phys_active = (ep >= PHYS_LOSS_WARMUP_EPOCH)
    lam_dag_warm = 0.0                                  # A_dag fixed, no warmup
    _current_gate = _gate_schedule(ep)                  # always 1.0 (set in Cell 7)
    _prev_gate    = _gate_schedule(ep - 1)              # always 1.0

    epoch_losses = {'mse': [], 'dag': [], 'l1': [], 'r10mm': [], 'rx1day': [], 'cdd': [], 'cc': [], 'causal_frac': [], 'gate_mean': []}
    _n_batches_done = 0
    for sample in train_dataset:
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        optimizer_s1.zero_grad(set_to_none=True)

        mu_A, mu_total, mu_B, gate, baseline_log, target_residual = _stage1_forward(batch)

        # causal_frac monitor (ML expert : go/no-go criterion for DAG learnable).
        # = ||mu_A|| / (||mu_A|| + ||mu_B||). Below 0.05 at ep 5 means Path A
        # (causal RCN) is architecturally cosmetic and UNet dual_path dominates.
        with torch.no_grad():
            _muA_n = float(mu_A.detach().pow(2).mean().sqrt())
            _muB_n = float(mu_B.detach().pow(2).mean().sqrt())
            epoch_losses['causal_frac'].append(_muA_n / (_muA_n + _muB_n + 1e-8))
            # Math expert monitor : gate.mean() = fraction of Path B (UNet direct LR->HR)
            # in fusion mu_total = g*mu_B + (1-g)*mu_A.
            # g ~ 0  => Path A (causal RCN) DOMINATES (healthy)
            # g ~ 1  => Path A BYPASSED, struct_W1/W2 grad=0 => DEAD ENCODER failure
            # Healthy zone: g.mean() in [0.05, 0.80]. Alert if > 0.90.
            epoch_losses['gate_mean'].append(float(gate.detach().mean()))

        # Core MSE on residual (HR - baseline) -- FIX NaN propagation (5-expert validated)
        valid = torch.isfinite(target_residual)
        target_residual_safe = torch.nan_to_num(target_residual, nan=0.0)
        diff = (mu_total - target_residual_safe) ** 2

        # P1 tail-weight (5-expert hyperplan, Climat blessing) :
        # Without this, mu_HR converges to a flat climatology under-predicting
        # Southern Alps/Westland orographic extremes. Estimator :
        #   loss_mse = sum(w * diff * valid) / sum(w * valid)
        # Horvitz-Thompson-style weighted MSE, consistent and essentially unbiased.
        hr_log_recon_s1 = baseline_log + target_residual_safe
        tail_w_s1 = _tail_weight(
            hr_log_recon_s1,
            tau95_mm=STAGE1_TAIL_TAU_P95,
            tau99_mm=STAGE1_TAIL_TAU_P99,
            w95=STAGE1_TAIL_W_P95,
            w99=STAGE1_TAIL_W_P99,
        ).detach()

        diff_weighted = diff * tail_w_s1
        diff_masked = torch.where(valid, diff_weighted, torch.zeros_like(diff_weighted))
        w_eff = tail_w_s1 * valid.float()
        loss_mse = diff_masked.sum() / w_eff.sum().clamp_min(1.0)
        assert torch.isfinite(loss_mse).item(), f'loss_mse NaN/Inf at ep={ep}'

        # DAG prior + L1 sparsity (5-expert AdamW fix : DAG prior now handled by
        # decoupled weight_decay on optimizer A_dag param group, NOT in loss_total).
        # loss_dag below is computed for MONITORING only (logging consistency).
        from torch.nn.parameter import UninitializedParameter
        _rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
        loss_dag = torch.tensor(0.0, device=DEVICE)
        loss_l1  = torch.tensor(0.0, device=DEVICE)
        # Option Z : A_dag is FIXED. All DAG losses (loss_dag L2 prior, loss_l1
        # sparsity, loss_dagma NOTEARS acyclicity) are unnecessary and would have
        # zero gradient anyway (A_dag.requires_grad=False). Kept as monitoring
        # zeros for log consistency.
        loss_dag = torch.tensor(0.0, device=DEVICE)
        loss_l1  = torch.tensor(0.0, device=DEVICE)

        # loss_total : MSE drives the backward. Phys losses added below if active.
        loss_total = loss_mse

        # Physical losses (after warmup)
        loss_r10  = torch.tensor(0.0, device=DEVICE)
        loss_rx1d = torch.tensor(0.0, device=DEVICE)
        loss_cdd  = torch.tensor(0.0, device=DEVICE)
        loss_cc   = torch.tensor(0.0, device=DEVICE)
        if phys_active:
            # Full HR pred in log1p space -- FIX NaN propagation (5-expert validated)
            # target_residual contains NaN where HR has QC gaps. Sanitize before
            # the sum to keep the gradient finite ; physical losses (R10mm, Rx1d,
            # CDD) all chain through (full_pred, full_tgt), so they would also
            # carry NaN gradients without this fix.
            target_residual_safe = torch.nan_to_num(target_residual, nan=0.0)
            full_pred = baseline_log + mu_total   # residual reconstruction (no delta yet at Stage 1)
            full_tgt  = baseline_log + target_residual_safe
            loss_r10  = LAMBDA_R10MM  * loss_R10mm(full_pred, full_tgt, valid_mask=valid.float())
            loss_rx1d = LAMBDA_RX1DAY * loss_Rx1day(full_pred, full_tgt)
            loss_cdd  = LAMBDA_CDD    * loss_CDD(full_pred, full_tgt)
            loss_total = loss_total + loss_r10 + loss_rx1d + loss_cdd
            # CC loss : skip if T_850 not in batch (would need to thread it through)
            # For now, document as TODO -- requires lr_grid to carry T_850 explicitly

        # Option Z : A_dag is frozen, so GRAD-DIAG (autograd.grad on A_dag) was
        # removed -- it would always return None for frozen params, generating
        # log noise. The Stage 1 grad flow is purely encoder + RCN MLPs +
        # regression_head + dual_path UNet (all standard params).

        loss_total.backward()
        # Option Z : A_dag.requires_grad=False so A_dag.grad is always None.
        # Targeted clip removed. Global clip on stage1_params remains.
        torch.nn.utils.clip_grad_norm_(stage1_params, 1.0)
        optimizer_s1.step()

        # Option Z : A_dag fixed, no projections needed (no possible collapse).

        epoch_losses['mse'].append(float(loss_mse.detach()))
        epoch_losses['dag'].append(float(loss_dag.detach()))
        epoch_losses['l1'].append(float(loss_l1.detach()))
        epoch_losses['r10mm'].append(float(loss_r10.detach()))
        epoch_losses['rx1day'].append(float(loss_rx1d.detach()))
        epoch_losses['cdd'].append(float(loss_cdd.detach()))
        epoch_losses['cc'].append(float(loss_cc.detach()))
        _n_batches_done += 1
        if SMOKE_MODE and _n_batches_done >= 30:
            break   # SMOKE : process only 30 train batches per epoch

    ep_time = time.time() - _t0
    avg = {k: float(np.mean(v)) if v else 0.0 for k, v in epoch_losses.items()}
    # FIX BUG NaN (5-expert) : monitor A_dag stats each epoch
    if hasattr(_rcn_core, 'A_dag') and torch.isfinite(_rcn_core.A_dag).all():
        with torch.no_grad():
            _a_norm = float(_rcn_core.A_dag.abs().mean())
            _a_max  = float(_rcn_core.A_dag.abs().max())
            # NOTEARS acyclicity (Recherche reviewer requirement) : h(A) = tr(e^{A o A}) - d.
            # On 8x8 matrix this is essentially free. h(A) = 0 <=> acyclic.
            _A_squared = _rcn_core.A_dag * _rcn_core.A_dag
            _h_A_dag = float(torch.linalg.matrix_exp(_A_squared).diagonal().sum() - _A_squared.shape[0])
    else:
        _a_norm = _a_max = _h_A_dag = float('nan')
    _avg_causal_frac = float(np.mean(epoch_losses['causal_frac'])) if epoch_losses['causal_frac'] else float('nan')
    _avg_gate_mean = float(np.mean(epoch_losses['gate_mean'])) if epoch_losses.get('gate_mean') else float('nan')
    print(f'[ep{ep}/{STAGE1_EPOCHS}] mse={avg["mse"]:.4f} dag={avg["dag"]:.4f} l1={avg["l1"]:.4f} '
          f'r10={avg["r10mm"]:.4f} rx1d={avg["rx1day"]:.4f} cdd={avg["cdd"]:.4f} '
          f'(phys={phys_active} wd_dag={lam_dag_warm:.3f} gate_sched={_current_gate:.2f}) '
          f'A_dag.abs(): mean={_a_norm:.4f} max={_a_max:.4f} h(A)={_h_A_dag:.3e} '
          f'causal_frac={_avg_causal_frac:.3f} dual_gate={_avg_gate_mean:.3f} '
          f'time={ep_time:.0f}s n_batches={_n_batches_done}')

    # GLOBAL KILL-SWITCH (5/5-expert post-collapse-90c3519, IA+E specification) :
    # Original guard fired only at ep==5 as a WARN. Replaced with a global guard that
    # raises on 2 CONSECUTIVE epochs of causal_frac < 0.05 from ep>=5. Rationale (Math) :
    # the degenerate FusionGate fixed point (g~0 + ||mu_A||~0) is a BASIN driven by gate
    # detach feedback on MSE alone -- any perturbation (phys, lambda drift, init noise)
    # can re-trigger it. 2-consecutive guard avoids single-epoch noise false-positive.
    # Saves .collapse.pt for forensic analysis before raising.
    if not hasattr(_stage1_forward, '_collapse_streak'):
        _stage1_forward._collapse_streak = 0
    if ep >= 5:
        if _avg_causal_frac < 0.05:
            _stage1_forward._collapse_streak += 1
            if _stage1_forward._collapse_streak >= 2:
                _collapse_path = CKPT_STAGE1_LAST.with_suffix('.collapse.pt')
                try:
                    CKPT_STAGE1_LAST.parent.mkdir(parents=True, exist_ok=True)
                    torch.save({
                        'ep': ep, 'causal_frac': _avg_causal_frac,
                        'dual_gate': _avg_gate_mean,
                        'history': stage1_history,
                        'streak': _stage1_forward._collapse_streak,
                    }, _collapse_path)
                    print(f'  [collapse debug] saved {_collapse_path}')
                except Exception as _ce:
                    print(f'  [collapse debug] save failed : {_ce}')
                raise RuntimeError(
                    f'[STAGE 1 COLLAPSE ABORT ep {ep}] causal_frac={_avg_causal_frac:.4f} '
                    f'< 0.05 for {_stage1_forward._collapse_streak} consecutive epochs. '
                    f'Path A (RCN-mu_A) magnitude collapsed -> FusionGate basin minimum. '
                    f'Stage 1 cache would be useless (mu_HR ~= 0). Reduce '
                    f'LAMBDA_R10MM/LAMBDA_RX1DAY/LAMBDA_CDD or set PHYS_LOSS_WARMUP_EPOCH '
                    f'> STAGE1_EPOCHS to disable phys losses entirely.'
                )
            else:
                print(f'[WARN ep{ep}] causal_frac={_avg_causal_frac:.4f} < 0.05 '
                      f'(streak {_stage1_forward._collapse_streak}/2). Next epoch decides '
                      f'abort vs recovery.')
        else:
            if _stage1_forward._collapse_streak > 0:
                print(f'  [collapse guard] streak reset (causal_frac recovered to {_avg_causal_frac:.4f})')
            _stage1_forward._collapse_streak = 0

    # Option Z : A_dag FIXED, no collapse possible by construction.

    stage1_history.append({
        'epoch': ep, 'epoch_time_s': ep_time, 'losses': avg,
        'lambda_l1': lam_l1, 'phys_active': phys_active,
        'n_batches': _n_batches_done,
    })

    # Save Stage 1 checkpoint each epoch
    payload_s1 = {
        'epoch': ep,
        'encoder_state_dict': encoder.state_dict(),
        'rcn_cell_state_dict': rcn_cell.state_dict(),
        'regression_head_state_dict': regression_head.state_dict(),
        'dual_path_state_dict': dual_path.state_dict(),
        'alpha_logit': alpha_logit.detach().cpu(),
        'optimizer_state_dict': optimizer_s1.state_dict(),
        'history': stage1_history,
        'pre_reg': PRE_REG_RECORD,
        'option_z': True,   # marker : Option Z pipeline (A_dag fixed Trenberth)
    }
    CKPT_STAGE1_LAST.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload_s1, CKPT_STAGE1_LAST)

    if SMOKE_MODE and ep >= STAGE1_EPOCHS:
        break

# Freeze Stage 1 after training (skip UninitializedParameter)
from torch.nn.parameter import UninitializedParameter as _UP
for m in [encoder, rcn_cell, regression_head, dual_path]:
    for p in m.parameters():
        if isinstance(p, _UP):
            continue
        p.requires_grad_(False)
    m.eval()

# Print final A_dag stats (Q_phys interpretability)
_rcn_core = rcn_cell._orig_mod if hasattr(rcn_cell, '_orig_mod') else rcn_cell
if hasattr(_rcn_core, 'A_dag'):
    _A = _rcn_core.A_dag.detach()
    print(f'[Cell 8] Final A_dag : shape={tuple(_A.shape)} norm={_A.norm():.4f} '
          f'asymmetry={(_A - _A.T).abs().mean():.4f}')

print(f'[Cell 8] Stage 1 training complete. Checkpoint : {CKPT_STAGE1_LAST}')

# M7 fix (IA MEDIUM) : free Stage 1 optimizer (Adam moments ~ 2x param size in fp32)
# and clear CUDA cache so Stage 2 build does not OOM at 80 GB A100. Without this,
# the lingering optimizer_s1 + 3 EMA + 50M diff_decoder peaked at OOM.
try:
    del optimizer_s1, stage1_params
except NameError:
    pass
import gc as _gc; _gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'[Cell 8] GPU memory cleared (optimizer_s1 freed, empty_cache)')


In [ ]:
# === Cell 9 : Precompute mu_total cache (Stage 1 FROZEN forward) ===
# One-time pass on the train_dataset to cache (mu_total, baseline_log, delta_target).
# Used by Stage 2 training (cached -> 70x faster than recomputing Stage 1 each batch).
import time

if STAGE1_CACHE_PATH.exists() and not SMOKE_MODE:
    print(f'[Cell 9] Cache exists, loading : {STAGE1_CACHE_PATH}')
    cache = torch.load(STAGE1_CACHE_PATH, map_location='cpu', weights_only=False)
    cache = {k: v.contiguous().clone() for k, v in cache.items()}
else:
    print(f'[Cell 9] Precomputing mu_total cache from train_dataset (SMOKE={SMOKE_MODE})...')
    _t0 = time.time()
    mu_list, base_list, delta_list, valid_list = [], [], [], []
    _count = 0
    for sample in train_dataset:
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        with torch.no_grad():
            mu_A, mu_total, mu_B, gate, baseline_log, target_residual = _stage1_forward(batch)
            delta_target = target_residual - mu_total
            valid = torch.isfinite(target_residual)
        mu_list.append(mu_total.squeeze(0).cpu())
        base_list.append(baseline_log.squeeze(0).cpu())
        delta_list.append(torch.nan_to_num(delta_target, nan=0.0).squeeze(0).cpu())
        valid_list.append(valid.squeeze(0).cpu())
        _count += 1
        if SMOKE_MODE and _count >= 30: break
        if _count % 500 == 0:
            print(f'  cached {_count} samples ({(time.time()-_t0)/60:.1f} min)')
    cache = {
        'mu_HR':        torch.stack(mu_list, dim=0),
        'baseline_log': torch.stack(base_list, dim=0),
        'delta_target': torch.stack(delta_list, dim=0),
        'valid_mask':   torch.stack(valid_list, dim=0),
    }

    # Option Z : A_dag fixed, no cache poison risk. Keep correlation diagnostic.
    import numpy as _np
    _mu_flat = mu_list[0].numpy().flatten()
    # L4 fix (ML LOW) : add mu_list[0] back to recover HR_log. Previously
    # we computed corr(mu, HR-mu) which is anti-correlated by construction
    # for a good Stage 1 (mu tracks HR), the diagnostic was misleading.
    _tgt_flat = (base_list[0].numpy() + delta_list[0].numpy() + mu_list[0].numpy()).flatten()
    _mask_flat = valid_list[0].numpy().flatten().astype(bool)
    if _mask_flat.sum() > 100:
        _corr = float(_np.corrcoef(_mu_flat[_mask_flat], _tgt_flat[_mask_flat])[0, 1])
        if not _np.isnan(_corr):
            print(f'[Cell 9] mu_HR/target correlation = {_corr:.3f} (sample[0])')
    STAGE1_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(cache, STAGE1_CACHE_PATH)
    print(f'[Cell 9] Cache built in {time.time()-_t0:.1f}s : {STAGE1_CACHE_PATH}')

# Integrity check (Expert IA from previous audit)
for k in ('mu_HR', 'baseline_log', 'delta_target'):
    if not torch.isfinite(cache[k]).all():
        raise RuntimeError(f'cache[{k!r}] contains NaN/Inf -- delete and rerun')

# Stage 1 skill diagnostic (GLOBAL, all samples) -- replaces the L4 sample[0]-only print
# Computes corr(mu_total, target_residual). Bounded by [-1, 1] (vs L4 corr(mu, HR_log)
# which is plafonded ~0.30 by baseline variance domination).
import numpy as _np_diag
_m = cache['mu_HR'].numpy().flatten()
_d = cache['delta_target'].numpy().flatten()   # = nan_to_num(target_residual - mu_total)
_v = cache['valid_mask'].numpy().flatten().astype(bool)
_residual_recon = (_m + _d)[_v]                 # = target_residual on valid pixels
_mu_flat = _m[_v]
_finite = _np_diag.isfinite(_mu_flat) & _np_diag.isfinite(_residual_recon)
if _finite.sum() > 100:
    _corr_skill = float(_np_diag.corrcoef(_mu_flat[_finite], _residual_recon[_finite])[0, 1])
    print(f'[Cell 9] corr(mu_total, target_residual) = {_corr_skill:.3f}   (GLOBAL, N_valid={_finite.sum():,})')
    print(f'         >0.5 = excellent | 0.3-0.5 = OK | 0.2-0.3 = marginal | <0.2 = RESTART')
else:
    print(f'[Cell 9] corr(mu_total, target_residual) : skipped (only {_finite.sum()} valid finite pixels)')

N_CACHE = int(cache['mu_HR'].shape[0])
print(f'[Cell 9] Cache shapes :')
for k, v in cache.items():
    print(f'  {k} = {tuple(v.shape)}')
print(f'[Cell 9] Total cached samples : {N_CACHE}')

# Cached dataset for Stage 2
class _CachedDataset(torch.utils.data.Dataset):
    def __init__(self, cache, indices=None):
        self.mu_HR = cache['mu_HR']
        self.baseline_log = cache['baseline_log']
        self.delta_target = cache['delta_target']
        self.valid_mask = cache['valid_mask']
        self.indices = indices if indices is not None else list(range(len(self.mu_HR)))
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        idx = self.indices[i]
        return {
            'mu_HR': self.mu_HR[idx],
            'baseline_log': self.baseline_log[idx],
            'delta_target': self.delta_target[idx],
            'valid_mask': self.valid_mask[idx],
        }

# FIX BUG #2 (ML) : Build a SEPARATE val cache from val_dataset (no leakage).
# VAL_CACHE_PATH moved to Cell 2 (referenced by Cell 8 defensive resume check)
if VAL_CACHE_PATH.exists() and not SMOKE_MODE:
    print(f'[Cell 9] Val cache exists, loading : {VAL_CACHE_PATH}')
    val_cache = torch.load(VAL_CACHE_PATH, map_location='cpu', weights_only=False)
    val_cache = {k: v.contiguous().clone() for k, v in val_cache.items()}
else:
    print(f'[Cell 9] Precomputing val cache (separate from train)...')
    _t1 = time.time()
    _mu_v, _base_v, _delta_v, _valid_v = [], [], [], []
    _count_v = 0
    for sample in val_dataset:
        batch = convert_sample_to_batch(sample, builder, DEVICE)
        with torch.no_grad():
            _, mu_total_v, _, _, baseline_log_v, target_residual_v = _stage1_forward(batch)
            delta_target_v = target_residual_v - mu_total_v
            valid_v = torch.isfinite(target_residual_v)
        _mu_v.append(mu_total_v.squeeze(0).cpu())
        _base_v.append(baseline_log_v.squeeze(0).cpu())
        _delta_v.append(torch.nan_to_num(delta_target_v, nan=0.0).squeeze(0).cpu())
        _valid_v.append(valid_v.squeeze(0).cpu())
        _count_v += 1
        if SMOKE_MODE and _count_v >= 6: break
    val_cache = {
        'mu_HR':        torch.stack(_mu_v,    dim=0),
        'baseline_log': torch.stack(_base_v,  dim=0),
        'delta_target': torch.stack(_delta_v, dim=0),
        'valid_mask':   torch.stack(_valid_v, dim=0),
    }
    VAL_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(val_cache, VAL_CACHE_PATH)
    print(f'[Cell 9] Val cache : N={_count_v} in {time.time()-_t1:.1f}s')

train_cached = _CachedDataset(cache, list(range(N_CACHE)))
val_cached   = _CachedDataset(val_cache, list(range(int(val_cache['mu_HR'].shape[0]))))
train_cached_dataloader = torch.utils.data.DataLoader(
    train_cached, batch_size=STAGE2_BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=False, drop_last=False,
)
val_cached_dataloader = torch.utils.data.DataLoader(
    val_cached, batch_size=STAGE2_BATCH_SIZE, shuffle=False,
    num_workers=0, pin_memory=False, drop_last=False,
)
print(f'[Cell 9] Stage 2 dataloaders : train={len(train_cached)}  val={len(val_cached)}  bs={STAGE2_BATCH_SIZE}')


In [ ]:
# === Cell 10 : Stage 2 build (UNet 50M EDM + Dispersive hook + multi-EMA) ===
import copy
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

# Probe HR channels
_probe = next(iter(val_dataset))
hr_channels = int(_probe['residual'].shape[1])

UNET_KW = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in UNET_KW and isinstance(UNET_KW[_k], list):
        UNET_KW[_k] = tuple(UNET_KW[_k])
UNET_KW['projection_class_embeddings_input_dim'] = num_vars * int(CONFIG.diffusion.conditioning_dim)

edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))

def _build_stage2_decoder():
    d = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=int(CONFIG.diffusion.height), width=int(CONFIG.diffusion.width),
        unet_kwargs=UNET_KW,
        scheduler_type=str(CONFIG.diffusion.scheduler_type),
        # FIX BUG #3 (ML+IA) : grad_checkpointing fires hook 2x (forward + recompute) ;
        # disable when DISPERSIVE_LAMBDA > 0 to avoid stale mid-block features.
        use_gradient_checkpointing=(DISPERSIVE_LAMBDA == 0.0),
        conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
        anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
        edm_config=edm_cfg, causal_concat=True,
    ).to(DEVICE)
    d.edm_config.sigma_data = float(SIGMA_DATA_NEW)
    return d

# Live decoder (the one being trained)
diff_decoder = _build_stage2_decoder()
n_params = sum(p.numel() for p in diff_decoder.parameters())
print(f'[Cell 10] Stage 2 decoder built : {n_params:,} params (sigma_data={SIGMA_DATA_NEW})')

# Multi-EMA models (Expert ML : 3 EMA decays + post-hoc sweep at eval)
ema_decoders = []
for decay in EMA_DECAYS:
    ema = _build_stage2_decoder()
    ema.load_state_dict(diff_decoder.state_dict())
    ema.eval()
    for p in ema.parameters(): p.requires_grad_(False)
    ema._ema_decay = decay
    ema._ema_step_counter = 0
    ema_decoders.append(ema)
print(f'[Cell 10] Multi-EMA decoders built : decays = {EMA_DECAYS}')

# ---- Dispersive Loss hook on UNet mid-block (Wang & He 2025) ----
# Captures mid-block features for the dispersive penalty term.
dispersive_features = {'mid': None}

def _dispersive_hook(module, inputs, output):
    # Module output is a Tensor (mid-block residual). Store a detached-noncausal handle.
    if isinstance(output, tuple):
        feat = output[0]
    else:
        feat = output
    dispersive_features['mid'] = feat

# Attach hook to UNet mid_block (diffusers UNet2DConditionModel exposes .mid_block)
try:
    _mid_block = diff_decoder.unet.mid_block
    _disp_handle = _mid_block.register_forward_hook(_dispersive_hook)
    print(f'[Cell 10] Dispersive Loss hook registered on mid_block (lambda={DISPERSIVE_LAMBDA}, tau={DISPERSIVE_TAU})')
except AttributeError:
    print(f'[Cell 10] WARNING : unet.mid_block not found, Dispersive Loss disabled')
    _disp_handle = None

def loss_dispersive(features, tau=DISPERSIVE_TAU):
    """Dispersive Loss (Wang & He 2025, arXiv 2506.09027).
    Encourages batch-wise feature diversity via contrastive repulsion.
    L = -log(E[K(z_i, z_j) / tau])  where K is a Gaussian kernel.
    """
    if features is None or features.dim() < 2:
        return torch.tensor(0.0, device=DEVICE)
    B = features.shape[0]
    if B < 2:
        return torch.tensor(0.0, device=DEVICE)
    flat = features.flatten(start_dim=1)
    # Pairwise distance (cosine-like via normalized dot product)
    flat = flat / (flat.norm(dim=1, keepdim=True) + 1e-8)
    sim = (flat @ flat.T) / tau   # [B, B]
    # Off-diagonal repulsion : we want low pairwise similarity
    mask = ~torch.eye(B, dtype=torch.bool, device=sim.device)
    off_diag = sim[mask]
    # -log(softmax-like form) : encourages off-diag to be small
    return torch.logsumexp(off_diag.view(B, B-1), dim=1).mean()

# ---- Min-SNR-gamma weighting (Hang ICCV 2023) ----
def _min_snr_weight(sigma, sigma_data, gamma):
    """w_min_snr(sigma) = min(SNR, gamma) / SNR  where SNR = sigma_d^2 / sigma^2.
    Final weight = w_min_snr * lambda_karras (combined per Math expert)."""
    snr = (sigma_data / sigma.clamp_min(1e-8)) ** 2
    return (torch.clamp(snr, max=gamma) / snr).view(-1, 1, 1, 1)

# ---- Tail weight (4, 12) on extreme pixels ----
def _tail_weight(hr_log_recon, tau95_mm=15.0, tau99_mm=35.0, w95=TAIL_WEIGHT_P95, w99=TAIL_WEIGHT_P99):
    import math
    tau95 = math.log1p(tau95_mm)
    tau99 = math.log1p(tau99_mm)
    return (1.0 + (w95 - 1.0) * (hr_log_recon > tau95).float()
                + (w99 - w95) * (hr_log_recon > tau99).float())

print(f'[Cell 10] Min-SNR weighting ready (gamma={MIN_SNR_GAMMA})')
print(f'[Cell 10] Tail weight ready ({TAIL_WEIGHT_P95}, {TAIL_WEIGHT_P99}) on pixels >15/>35 mm/day')


In [ ]:
# === Cell 11 : Stage 2 training (Min-SNR + tail + Dispersive + FACL + cond_drop + multi-EMA) ===
# FACL (Yang NeurIPS 2024) for spectral consistency : FAL drives RAPSD,
# FCL drives spatial phase coherence (critical for Westland orographic precip).
from st_cdgm.training.spectral_loss import facl_loss, facl_components
# M6 fix (IA MEDIUM) : guard SMOKE_MODE config where STAGE2_EPOCHS can be small.
# Without this, FACL_WARMUP_EPOCHS==STAGE2_EPOCHS would trigger abort on the
# last SMOKE epoch and ALWAYS crash SMOKE runs.
assert FACL_WARMUP_EPOCHS < STAGE2_EPOCHS, (
    f'FACL_WARMUP_EPOCHS={FACL_WARMUP_EPOCHS} must be < STAGE2_EPOCHS={STAGE2_EPOCHS} '
    f'(otherwise abort gate fires on last SMOKE epoch and always crashes)'
)
# Inline implementation -- adapts train_epoch_stage2_cached pattern with all the fixes.
from st_cdgm.models.edm_preconditioner import sample_training_sigma
import time

# Optimizer (Stage 2 live + alpha_logit)
stage2_params = list(diff_decoder.parameters()) + [alpha_logit]
optimizer_s2 = torch.optim.AdamW(
    stage2_params,
    lr=STAGE2_LR, betas=(0.9, 0.999), weight_decay=STAGE2_WEIGHT_DECAY,
)

stage2_history = []
start_epoch_s2 = 1

# Resume support
if CKPT_STAGE2_LAST.exists() and not SMOKE_MODE:
    print(f'[Cell 11] RESUME from {CKPT_STAGE2_LAST}')
    _ck = torch.load(CKPT_STAGE2_LAST, map_location=DEVICE, weights_only=False)
    diff_decoder.load_state_dict(_ck['diffusion_state_dict'])
    if 'ema_state_dicts' in _ck:
        for i, ema in enumerate(ema_decoders):
            if i < len(_ck['ema_state_dicts']):
                ema.load_state_dict(_ck['ema_state_dicts'][i])
    if _ck.get('alpha_logit') is not None:
        alpha_logit.data = _ck['alpha_logit'].to(DEVICE)
    if _ck.get('optimizer_state_dict') is not None:
        try: optimizer_s2.load_state_dict(_ck['optimizer_state_dict'])
        except Exception as e: print(f'  optimizer resume failed : {e}')
    start_epoch_s2 = int(_ck.get('epoch', 0)) + 1
    stage2_history = list(_ck.get('history', []))
    print(f'  Resumed at epoch {start_epoch_s2}/{STAGE2_EPOCHS}')
    # L3 fix (ML LOW) : silent no-op guard when CKPT already complete.
    if start_epoch_s2 > STAGE2_EPOCHS:
        print(f'  Stage 2 already complete in checkpoint -- skipping loop')
    # H3 fix (IA HIGH) : warn if resume position skips the FACL abort gate.
    elif start_epoch_s2 > max(1, FACL_WARMUP_EPOCHS):
        print(f'  [Cell 11 WARN] start_epoch_s2={start_epoch_s2} > FACL abort gate '
              f'({max(1, FACL_WARMUP_EPOCHS)}) -- FACL/EDM ratio safety check '
              f'will NOT re-fire on this resume. If the previous run had FACL '
              f'instability, manually delete CKPT to re-run from epoch 1.')

print(f'[Cell 11] Stage 2 training : epochs {start_epoch_s2}..{STAGE2_EPOCHS} (SMOKE={SMOKE_MODE})')

def _ema_update_all(decoders, decays, live_model):
    """Multi-EMA update : in-place mul + add on params, copy on buffers."""
    with torch.no_grad():
        for ema, decay in zip(decoders, decays):
            for p_e, p_l in zip(ema.parameters(), live_model.parameters()):
                p_e.data.mul_(decay).add_(p_l.data, alpha=1.0 - decay)
            for b_e, b_l in zip(ema.buffers(), live_model.buffers()):
                b_e.data.copy_(b_l.data)
            ema._ema_step_counter += 1

# H4 fix (IA HIGH) : try/finally guarantees the dispersive hook is detached even
# if the FACL abort raises. Without this, the hook leaked GPU memory across
# kernel restarts (OOM at 80 GB A100 documented in prior runs).
try:
    for ep in range(start_epoch_s2, STAGE2_EPOCHS + 1):
        _t0 = time.time()
        diff_decoder.train()

        losses_log = {'edm': [], 'disp': [], 'alpha_reg': [], 'total': [], 'facl': [], 'fal': [], 'fcl': [], 'fraction_valid': []}
        n_batches_seen = 0

        for batch in train_cached_dataloader:
            mu_HR = batch['mu_HR'].to(DEVICE, non_blocking=True)
            baseline_log = batch['baseline_log'].to(DEVICE, non_blocking=True)
            delta_target = batch['delta_target'].to(DEVICE, non_blocking=True)
            # IA expert fix : load valid_mask to exclude NaN/QC pixels from loss
            # (without this, delta_target=0 on QC pixels biases model toward
            # under-prediction in those regions).
            valid_mask = batch['valid_mask'].to(DEVICE, non_blocking=True).float()

            # Conditioning dropout (CFG compatibility, CorrDiff Nature CEE 2025)
            # When mu_HR is dropped, target also adjusted (Ho & Salimans branch correctness)
            mu_HR_used = mu_HR
            delta_target_used = delta_target
            if COND_DROPOUT_P > 0:
                B = mu_HR.shape[0]
                drop_mask = (torch.rand(B, device=DEVICE) < COND_DROPOUT_P).view(B, 1, 1, 1)
                if drop_mask.any():
                    mu_HR_used = torch.where(drop_mask, torch.zeros_like(mu_HR), mu_HR)
                    delta_target_used = torch.where(drop_mask, delta_target + mu_HR, delta_target)

            optimizer_s2.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                B = delta_target_used.shape[0]
                sigma = sample_training_sigma(B, P_mean=edm_cfg.P_mean, P_std=edm_cfg.P_std,
                                                device=DEVICE, dtype=delta_target_used.dtype)
                noise = torch.randn_like(delta_target_used)
                y_noisy = delta_target_used + sigma.view(-1, 1, 1, 1) * noise

                # Forward EDM (this triggers the dispersive hook on mid_block)
                D_y = diff_decoder.forward_edm(
                    y_noisy, sigma, conditioning=None, conditioning_spatial=None,
                    mu_HR=mu_HR_used, baseline_log=baseline_log,
                )

                # FIX BUG #4 (ML+Math) : Min-SNR alone, NOT * Karras lambda.
                # Karras EDM2 + Hang ICCV 2023 specify Min-SNR REPLACES the standard lambda.
                min_snr_w = _min_snr_weight(sigma, edm_cfg.sigma_data, MIN_SNR_GAMMA)
                w_total = min_snr_w

                # ----- Tail weight on extreme pixels -----
                hr_log_recon = delta_target_used + mu_HR_used + baseline_log   # full HR in log1p
                tail_w = _tail_weight(hr_log_recon).detach()
                # Clip combined weight to avoid extreme values (Math : W_max=50)
                w_combined = (w_total * tail_w).clamp(max=50.0)

                # IA expert fix #4 : apply valid_mask to exclude NaN/QC pixels.
                # Without this, cached delta_target=0 on QC pixels biases the model
                # toward under-prediction on coastal/glacier regions.
                sq_err = (D_y - delta_target_used) ** 2
                _eff_w = w_combined * valid_mask
                loss_edm = (_eff_w * sq_err).sum() / _eff_w.sum().clamp_min(1.0)

                # ----- Dispersive Loss on mid-block features -----
                loss_disp = loss_dispersive(dispersive_features.get('mid', None))

                # ----- Alpha regularization -----
                alpha = torch.sigmoid(alpha_logit)
                loss_alpha = loss_alpha_reg(alpha)

                # ----- FACL : Fourier Amplitude + Correlation Loss (Yang NeurIPS 2024) -----
                # B3 (IA expert) : cast to .float() because rfft2 BF16 support is
                #   PyTorch-version-dependent under autocast. cuFFT promotes to fp32
                #   internally but explicit cast removes ambiguity. Cost negligible.
                # B2 (IA+Math)   : drop valid_mask kwarg ; delta_target_used has no NaN
                #   (cache pre-cleaned via Stage 1 nan_to_num), _zero_nans inside
                #   facl_loss handles any residual NaN defensively.
                # I2 (ML)        : FACL warmup over FACL_WARMUP_EPOCHS to prevent
                #   early-training domination when D_y is far from target (FCL ~ 1.0
                #   at init, can be 10-50x larger than edm loss).
                facl_warmup_factor = min(1.0, float(ep) / max(1, FACL_WARMUP_EPOCHS))
                # Math + ML expert : use facl_components for separate FAL+FCL logging
                # to enable diagnostic if either dominates. Reconstruct loss_facl manually.
                # M12 ROLLBACK (4/5 expert : ML+Math+Climat+IA vs Recherche dissent post-e823600 audit).
                # Math+ML : passing valid_mask creates an ASYMMETRIC perturbation : target is
                # already 0 over ocean (cache nan_to_num upstream), so FFT(target*mask) = FFT(target)
                # (no-op), but FFT(pred*mask) = FFT(pred) circ_conv FFT(mask), injecting sinc-sidelobe
                # leakage on pred side only. Model can minimize FAL by zeroing ocean output (trivial
                # shortcut, unrelated to land precip skill). Climat revised : Westland orographic
                # peak k~0.02 cyc/km is exactly where sinc spreads, predicted Rx1day -3..-8%, RAPSD
                # high-k -5..-15%. Recherche's KEEP argument (Yang 2024 zero-pad ~= mask) doesn't
                # transfer because Yang zero-pads BOTH sides symmetrically -- our cache already
                # zero-padded target, so M12 introduces a NEW pred-only asymmetry.
                # Future improvement (Climat) : Tukey/Hann taper at land-sea boundary (>40dB sinc
                # suppression) -- defer to post-FULL #1.
                _fal_val, _fcl_val = facl_components(D_y.float(), delta_target_used.float())
                loss_facl = (LAMBDA_FACL_FAL * facl_warmup_factor) * _fal_val \
                          + (LAMBDA_FACL_FCL * facl_warmup_factor) * _fcl_val

                loss_total = loss_edm + DISPERSIVE_LAMBDA * loss_disp + loss_alpha + loss_facl

            # H1 fix (ML HIGH) : NaN/Inf guard. Without this, a single corrupt batch (FACL
            # spike from a degenerate rfft2, dispersive collapse, etc.) propagates NaN through
            # backward(), corrupting BOTH the live model AND the 3 EMA copies AND the next
            # checkpoint -- silent training collapse with no abort signal.
            if not torch.isfinite(loss_total):
                print(f'[WARN ep{ep} batch{n_batches_seen}] loss_total={float(loss_total):.4f} '
                      f'(edm={float(loss_edm):.3f} disp={float(loss_disp):.3f} '
                      f'facl={float(loss_facl):.3f}) -- SKIPPING backward+step')
                optimizer_s2.zero_grad(set_to_none=True)
                n_batches_seen += 1
                continue
            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(stage2_params, STAGE2_GRADIENT_CLIP)
            optimizer_s2.step()

            # Multi-EMA update
            _ema_update_all(ema_decoders, EMA_DECAYS, diff_decoder)

            losses_log['edm'].append(float(loss_edm.detach()))
            losses_log['disp'].append(float(loss_disp.detach()))
            losses_log['alpha_reg'].append(float(loss_alpha.detach()))
            losses_log['facl'].append(float(loss_facl.detach()))
            losses_log['fal'].append(float(_fal_val.detach()))
            losses_log['fcl'].append(float(_fcl_val.detach()))
            losses_log['total'].append(float(loss_total.detach()))
            # M1 fix (Math MEDIUM) : monitor mask fraction ; FAL leakage from mask-shape DFT
            # scales with (1 - mean(valid_mask)). If this drops <0.5, FAL measures mask
            # geometry more than spectral skill -- training signal degraded.
            losses_log['fraction_valid'].append(float(valid_mask.detach().mean()))
            n_batches_seen += 1

            if SMOKE_MODE and n_batches_seen >= 10:
                break

        ep_time = time.time() - _t0
        avg = {k: float(np.mean(v)) if v else 0.0 for k, v in losses_log.items()}
        cur_alpha = float(torch.sigmoid(alpha_logit).detach())
        print(f'[ep{ep}/{STAGE2_EPOCHS}] edm={avg["edm"]:.4f} disp={avg["disp"]:.4f} '
              f'alpha_reg={avg["alpha_reg"]:.4f} total={avg["total"]:.4f} '
              f'alpha={cur_alpha:.4f} time={ep_time:.0f}s n_batches={n_batches_seen}')

        stage2_history.append({
            'epoch': ep, 'epoch_time_s': ep_time, 'losses': avg,
            'alpha': cur_alpha, 'n_batches': n_batches_seen,
        })

        # H2+M4 fix (IA+ML experts) : abort BEFORE checkpoint save, and AFTER warmup
        # so we measure FACL at full strength. Without these fixes:
        #   - abort at ep==1 measured 1/3 of full FACL (warmup) => missed scenarios
        #     where steady-state facl/edm ratio is 3-9x.
        #   - if abort raised at ep==1, checkpoint already saved with epoch=1,
        #     next resume would skip the check silently => silent failure path.
        # M10 fix (ML MEDIUM) : max(1, ...) so the abort still fires at ep=1 even when
        # FACL_WARMUP_EPOCHS=0 (no warmup), preventing latent config-sensitivity bypass.
        if (ep == max(1, FACL_WARMUP_EPOCHS) and
                len(losses_log['edm']) > 0 and len(losses_log['facl']) > 0):
            _mean_edm  = float(np.mean(losses_log['edm']))
            _mean_facl = float(np.mean(losses_log['facl']))
            if _mean_facl > 3.0 * max(_mean_edm, 1e-6):
                # M4 fix (IA MEDIUM) : persist debug payload BEFORE raise -- regular CKPT
                # is intentionally skipped for this epoch, so without this we lose all
                # diagnostic signal (history, per-batch losses) on the failing epoch.
                _abort_path = CKPT_STAGE2_LAST.with_suffix('.abort.pt')
                try:
                    CKPT_STAGE2_LAST.parent.mkdir(parents=True, exist_ok=True)
                    torch.save({
                        'history': stage2_history, 'losses_log_last_ep': losses_log,
                        'ep': ep, 'mean_edm': _mean_edm, 'mean_facl': _mean_facl,
                    }, _abort_path)
                    print(f'  [abort debug] saved {_abort_path}')
                except Exception as _abe:
                    print(f'  [abort debug] save failed : {_abe}')
                raise RuntimeError(
                    f'[ABORT ep {ep}] FACL dominates after warmup : '
                    f'mean(facl)={_mean_facl:.4f} > 3x mean(edm)={3*_mean_edm:.4f}. '
                    f'Reduce LAMBDA_FACL_FAL/FCL (current : {LAMBDA_FACL_FAL}, '
                    f'{LAMBDA_FACL_FCL}). Checkpoint NOT saved for this epoch.'
                )

        # Save Stage 2 checkpoint each epoch
        payload_s2 = {
            'epoch': ep,
            'diffusion_state_dict': diff_decoder.state_dict(),
            'ema_state_dicts': [ema.state_dict() for ema in ema_decoders],
            'ema_decays': EMA_DECAYS,
            'ema_step_counters': [ema._ema_step_counter for ema in ema_decoders],
            'alpha_logit': alpha_logit.detach().cpu(),
            'optimizer_state_dict': optimizer_s2.state_dict(),
            'history': stage2_history,
            'sigma_data': float(SIGMA_DATA_NEW),
            'pre_reg': PRE_REG_RECORD,
        }
        CKPT_STAGE2_LAST.parent.mkdir(parents=True, exist_ok=True)
        torch.save(payload_s2, CKPT_STAGE2_LAST)


        if SMOKE_MODE and ep >= STAGE2_EPOCHS:
            break

finally:
    # H4 : always detach hook to release GPU memory before next cell or restart.
    if _disp_handle is not None:
        try: _disp_handle.remove()
        except Exception: pass
    try:
        if isinstance(dispersive_features, dict):
            dispersive_features['mid'] = None
    except Exception:
        pass

# Set all to eval at the end
diff_decoder.eval()
for ema in ema_decoders:
    ema.eval()

print(f'[Cell 11] Stage 2 training complete')
print(f'[Cell 11] Final alpha = {float(torch.sigmoid(alpha_logit)):.4f}')
print(f'[Cell 11] Final EMA step counters : {[ema._ema_step_counter for ema in ema_decoders]}')
print(f'[Cell 11] Checkpoint : {CKPT_STAGE2_LAST}')


In [ ]:
# === Cell 12 : Sampling Phase 8 BS30 (N=64 x K=128 x 32 steps) avec post-hoc EMA sweep ===
import time

# Select EMA decoder via post-hoc sweep on a small validation set
# (Expert ML : choose the EMA decay that minimises val CRPS / RMSE)
def _ema_select_via_val(ema_list, val_loader, n_check=4):
    """Quick eval each EMA on a few val batches, return the one with lowest RMSE."""
    scores = []
    for i, ema in enumerate(ema_list):
        rmses = []
        with torch.no_grad():
            count = 0
            for batch in val_loader:
                if count >= n_check: break
                mu = batch['mu_HR'].to(DEVICE); base = batch['baseline_log'].to(DEVICE)
                tgt = batch['delta_target'].to(DEVICE)
                samples = []
                for _ in range(min(8, K_SAMPLES)):
                    out = ema.sample(
                        conditioning=None,
                        num_steps=N_STEPS_DIFF,
                        scheduler_type='edm_karras',
                        cfg_scale=0.0,
                        apply_constraints=False,
                        mu_HR=mu, baseline_log=base,
                    )
                    res = out.residual if hasattr(out, 'residual') else out
                    samples.append(res)
                pred = torch.stack(samples, dim=0).mean(dim=0)
                rmses.append(float(((pred - tgt) ** 2).mean().sqrt()))
                count += 1
        scores.append(np.mean(rmses) if rmses else float('inf'))
    best_i = int(np.argmin(scores))
    print(f'[Cell 12] EMA sweep RMSEs : {[f"{s:.4f}" for s in scores]}  best decay = {EMA_DECAYS[best_i]}')
    return ema_list[best_i], EMA_DECAYS[best_i]

best_ema, best_decay = _ema_select_via_val(ema_decoders, val_cached_dataloader, n_check=2 if SMOKE_MODE else 4)
print(f'[Cell 12] Selected EMA decoder (decay={best_decay}) for final BS30 eval')

# Now run full BS30 eval on test split.
# We iterate test_dataset (IterableDataset, fresh stream).
print(f'[Cell 12] Sampling : N={N_TEST_BATCHES} batches x K={K_SAMPLES} samples x {N_STEPS_DIFF} steps')
print(f'[Cell 12] Sampler = {SAMPLER_SCHEDULER}, cfg_scale = {CFG_SCALE}')
print(f'[Cell 12] Limited-Interval Guidance sigma in [{LIMITED_GUIDANCE_SIGMA_MIN}, {LIMITED_GUIDANCE_SIGMA_MAX}]')

_t0 = time.time()
test_pred_list = []   # K-mean per batch
test_target_list = []
test_mu_list = []
test_baseline_list = []
test_time_list = []   # for climate indices on full 730 days

# Use Stage 1 frozen + Stage 2 EMA selected
_count = 0
for sample in test_dataset:
    if _count >= N_TEST_BATCHES: break
    batch = convert_sample_to_batch(sample, builder, DEVICE)
    with torch.no_grad():
        mu_A, mu_total, mu_B, gate, baseline_log, target_residual = _stage1_forward(batch)
        samples = []
        for k in range(K_SAMPLES):
            torch.manual_seed(SEED + 1000 * _count + k)
            out = best_ema.sample(
                conditioning=None,
                num_steps=N_STEPS_DIFF,
                scheduler_type='edm_karras',  # fallback if dpm_solver++ not present
                cfg_scale=CFG_SCALE,
                apply_constraints=False,
                mu_HR=mu_total, baseline_log=baseline_log,
            )
            res = out.residual if hasattr(out, 'residual') else out
            samples.append(res)
        pred_delta = torch.stack(samples, dim=0).mean(dim=0)
    test_pred_list.append(pred_delta.detach().cpu())
    test_target_list.append(target_residual.detach().cpu())
    test_mu_list.append(mu_total.detach().cpu())
    test_baseline_list.append(baseline_log.detach().cpu())
    _t = sample.get('time', None)
    test_time_list.append(_t)
    _count += 1
    if _count % 4 == 0:
        print(f'  batch {_count}/{N_TEST_BATCHES}  elapsed={time.time()-_t0:.0f}s  '
              f'avg/batch={(time.time()-_t0)/_count:.1f}s', flush=True)

test_pred     = torch.cat(test_pred_list, dim=0)
test_target   = torch.cat(test_target_list, dim=0)
test_mu       = torch.cat(test_mu_list, dim=0)
test_baseline = torch.cat(test_baseline_list, dim=0)

# Full HR (log1p space) -- FIXED per Expert review BUG #1
# test_target from _stage1_forward is target_residual = HR_log - baseline (NOT delta).
# In Stage 2 training (Cell 11), the diffusion learns delta = HR - baseline - mu_HR.
# So reconstruction at inference must be : pred_full = baseline + mu + delta (alpha=1).
# alpha_logit is regularised toward 0.5 but never used as reconstruction weight in training,
# so we don't apply it here either (it remains a diagnostic of the learned conditioning strength).
# FIX BUG #8 (Climat) : alpha_logit is diagnostic-only ; training used alpha=1 implicitly.
alpha_final = float(torch.sigmoid(alpha_logit).detach())
pred_full_log   = test_baseline + test_mu + test_pred       # alpha=1
target_full_log = test_baseline + test_target               # HR_log = baseline + (HR-baseline)
print(f'[Cell 12] alpha_learned (conditioning strength diag) = {alpha_final:.4f}')
if abs(alpha_final - 0.5) > 0.2:
    print(f'[Cell 12] WARNING : alpha drifted to {alpha_final:.4f} (target=0.5) -- conditioning may have collapsed or saturated')
print(f'[Cell 12] reconstruction : pred = baseline + mu + delta_pred (alpha=1)')

# Convert to mm/day (Climat fix : - PRECIP_DELTA, clamp 500)
PRECIP_DELTA = 0.01
pred_mm   = (torch.expm1(pred_full_log)   - PRECIP_DELTA).clamp(min=0, max=500)
target_mm = (torch.expm1(target_full_log) - PRECIP_DELTA).clamp(min=0, max=500)

print(f'[Cell 12] Sampling done in {time.time()-_t0:.0f}s')
print(f'[Cell 12] alpha_final used in reconstruction = {alpha_final:.4f}')
print(f'[Cell 12] pred_full shape = {tuple(pred_full_log.shape)}')


In [ ]:
# === Cell 13 : Metriques (3 conventions F1, CSI, SEDI, FSS, CRPS + indices climatiques) ===
from scipy.ndimage import uniform_filter

# Load climatology + land_mask for ETCCDI eval
_clim = np.load(CLIM_PATH)
clim_p99_np = _clim['clim_p99']
clim_p95_np = _clim['clim_p95']
land_mask_np = _clim['land_mask']

# H5 fix (Climat HIGH) : assert clim_p99 unit is mm/day (NZ ETCCDI range 5-500 mm/day).
# If Cell 4 climatology was accidentally built on log1p instead of mm/day, the
# threshold is meaningless and F1@p99_etccdi reports a noise floor without warning.
_p99_finite = clim_p99_np[np.isfinite(clim_p99_np)]
assert _p99_finite.size > 0, 'clim_p99 all-NaN'
_p99_max = float(_p99_finite.max())
assert 5.0 < _p99_max < 500.0, (
    f'clim_p99 max={_p99_max:.2f} out of NZ ETCCDI range (5-500 mm/day) -- '
    f'check Cell 4 climatology built in mm/day, not log1p'
)

# ----- F1 / CSI / SEDI helper functions -----
def f1_pooled(pred, target, percentile=99.0):
    """Pooled full-grid, valid-only filter, strict > (Phase 6 audit fix)."""
    pf = pred.flatten(); tf = target.flatten()
    v = torch.isfinite(pf) & torch.isfinite(tf)
    pv = pf[v]; tv = tf[v]
    if tv.numel() < 100: return float('nan')
    thr = float(torch.quantile(tv, percentile / 100.0))
    # FIX BUG #7 : ETCCDI standard convention uses >=
    tp = ((pv >= thr) & (tv >= thr)).sum().item()
    fp = ((pv >= thr) & (tv < thr)).sum().item()
    fn = ((pv < thr) & (tv >= thr)).sum().item()
    if (tp + fp) == 0 or (tp + fn) == 0: return 0.0
    prec = tp / (tp + fp); rec = tp / (tp + fn)
    return (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0

def f1_etccdi_per_pixel(pred_mm, target_mm, land_mask, clim_p99):
    """ETCCDI per-pixel : seuil per-pixel = clim_p99[i,j]. Land only."""
    N = pred_mm.shape[0]
    pred_np = pred_mm.cpu().numpy().reshape(N, *clim_p99.shape)
    target_np = target_mm.cpu().numpy().reshape(N, *clim_p99.shape)
    valid_pix = land_mask & np.isfinite(clim_p99)
    if not valid_pix.any(): return {'f1': float('nan'), 'tp': 0, 'fp': 0, 'fn': 0}
    thr = clim_p99[None, :, :]
    pred_bin = pred_np >= thr
    target_bin = target_np >= thr
    valid_bcast = valid_pix[None, :, :]
    tp = int((pred_bin & target_bin & valid_bcast).sum())
    fp = int((pred_bin & ~target_bin & valid_bcast).sum())
    fn = int((~pred_bin & target_bin & valid_bcast).sum())
    if (tp + fp) == 0 or (tp + fn) == 0: return {'f1': 0.0, 'tp': tp, 'fp': fp, 'fn': fn}
    prec = tp / (tp + fp); rec = tp / (tp + fn)
    return {'f1': 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0, 'tp': tp, 'fp': fp, 'fn': fn}

def csi_sedi(pred_bin, target_bin):
    tp = int((pred_bin & target_bin).sum())
    fp = int((pred_bin & ~target_bin).sum())
    fn = int((~pred_bin & target_bin).sum())
    tn = int((~pred_bin & ~target_bin).sum())
    csi = tp / max(tp + fp + fn, 1)
    h = tp / max(tp + fn, 1); f = fp / max(fp + tn, 1)
    eps = 1e-12
    h = max(min(h, 1 - eps), eps); f = max(min(f, 1 - eps), eps)
    num = np.log(f) - np.log(h) - np.log(1 - f) + np.log(1 - h)
    den = np.log(f) + np.log(h) + np.log(1 - f) + np.log(1 - h)
    sedi = float(num / den) if den != 0 else float('nan')
    return csi, sedi

def fss_neighborhood(pred_bin, target_bin, n=9, mode='reflect'):
    P = pred_bin.astype(np.float32); Q = target_bin.astype(np.float32)
    Pf = np.stack([uniform_filter(P[i], size=n, mode=mode) for i in range(P.shape[0])])
    Qf = np.stack([uniform_filter(Q[i], size=n, mode=mode) for i in range(Q.shape[0])])
    mse  = float(((Pf - Qf) ** 2).mean())
    norm = float((Pf ** 2 + Qf ** 2).mean())
    return 1.0 - mse / max(norm, 1e-12)

# ----- Compute metrics on Phase 8 (live N=64 K=128 ensemble) -----
print('=' * 78)
print('Phase 8 BS30 metrics on TEST split')
print('=' * 78)

f1_p99_pool = f1_pooled(pred_full_log, target_full_log, 99.0)
f1_p95_pool = f1_pooled(pred_full_log, target_full_log, 95.0)
print(f'  F1@p99 (pooled, log1p)    = {f1_p99_pool:.4f}')
print(f'  F1@p95 (pooled, log1p)    = {f1_p95_pool:.4f}')

etccdi_p99 = f1_etccdi_per_pixel(pred_mm, target_mm, land_mask_np, clim_p99_np)
etccdi_p95 = f1_etccdi_per_pixel(pred_mm, target_mm, land_mask_np, clim_p95_np)
print(f'  F1@p99 (ETCCDI per-pixel) = {etccdi_p99["f1"]:.4f}  (TP={etccdi_p99["tp"]} FP={etccdi_p99["fp"]} FN={etccdi_p99["fn"]})')
print(f'  F1@p95 (ETCCDI per-pixel) = {etccdi_p95["f1"]:.4f}')

# CSI / SEDI / FSS at ETCCDI p99 threshold
pred_np = pred_mm.cpu().numpy().reshape(test_target.shape[0], H_HR, W_HR)
target_np = target_mm.cpu().numpy().reshape(test_target.shape[0], H_HR, W_HR)
valid_pix = land_mask_np & np.isfinite(clim_p99_np)
pred_bin = (pred_np >= clim_p99_np[None]) & valid_pix[None]
target_bin = (target_np >= clim_p99_np[None]) & valid_pix[None]
csi, sedi = csi_sedi(pred_bin, target_bin)
fss_9 = fss_neighborhood(pred_bin, target_bin, n=9, mode='reflect')
fss_25 = fss_neighborhood(pred_bin, target_bin, n=25, mode='reflect')
fss_51 = fss_neighborhood(pred_bin, target_bin, n=51, mode='reflect')
print(f'  CSI@p99 = {csi:.4f}  SEDI@p99 = {sedi:.4f}')
print(f'  FSS (n=9)={fss_9:.4f}  (n=25)={fss_25:.4f}  (n=51)={fss_51:.4f}')

# RMSE, MAE, Pearson on log1p full pred
_valid = torch.isfinite(target_full_log)
pf = pred_full_log[_valid]; tf = target_full_log[_valid]
rmse = float(((pf - tf) ** 2).mean().sqrt())
mae  = float((pf - tf).abs().mean())
pf_c = pf - pf.mean(); tf_c = tf - tf.mean()
pearson_global = float((pf_c * tf_c).sum() / ((pf_c.norm() * tf_c.norm()).clamp_min(1e-12)))
per_sample = []
for i in range(pred_full_log.shape[0]):
    vi = _valid[i]
    if vi.sum() < 2: continue
    pi = pred_full_log[i][vi]; ti = target_full_log[i][vi]
    pic = pi - pi.mean(); tic = ti - ti.mean()
    c = float((pic * tic).sum() / ((pic.norm() * tic.norm()).clamp_min(1e-12)))
    per_sample.append(c)
pearson_ps = float(np.mean(per_sample)) if per_sample else float('nan')
print(f'  RMSE = {rmse:.4f}  MAE = {mae:.4f}  Pearson_global = {pearson_global:.4f}  Pearson_ps = {pearson_ps:.4f}')

# Climate indices on the 64 batches (note : for full 730-day indices, would need full test pass)
# For SMOKE, we just compute on the sampled batches as proxy
pred_mm_np = pred_mm.cpu().numpy()      # shape (N, 1, H, W)
target_mm_np = target_mm.cpu().numpy()
# Squeeze channel dim so land_mask_np (H, W) broadcasts on the spatial axes.
if pred_mm_np.ndim == 4:
    pred_mm_np_2d   = pred_mm_np[:, 0]      # (N, H, W)
    target_mm_np_2d = target_mm_np[:, 0]
else:
    pred_mm_np_2d   = pred_mm_np
    target_mm_np_2d = target_mm_np

# RX1day : per-pixel max over the N sampled batches (approx of annual max when N << 730)
_p_max = np.nanmax(pred_mm_np_2d,   axis=0)   # (H, W)
_t_max = np.nanmax(target_mm_np_2d, axis=0)
rx1d_pred   = float(np.nanmean(_p_max[land_mask_np]))
rx1d_target = float(np.nanmean(_t_max[land_mask_np]))
rx1d_bias = rx1d_pred - rx1d_target
# H6 fix (Climat HIGH) : Rx1day is the per-pixel ANNUAL max ; with N_test=64 days
# we get a 64-day max, severely under-reported vs published Rx1day (NZ Westland
# ~150-250 mm/year). Bias preserved (both equally under-sampled) but absolute
# value is NOT ETCCDI Rx1day. Label the metric to reflect this.
_n_test_days = int(pred_mm_np_2d.shape[0])
rx1d_label = 'Rx1day' if _n_test_days >= 300 else f'Rx1day_{_n_test_days}d'

# R10mm count (per-pixel sum of days >= 10 mm/day) and CDD (per-pixel sum of days < 1 mm/day)
_p_r10 = (pred_mm_np_2d   >= 10).sum(axis=0)   # (H, W)
_t_r10 = (target_mm_np_2d >= 10).sum(axis=0)
r10_pred   = float(np.nanmean(_p_r10[land_mask_np]))
r10_target = float(np.nanmean(_t_r10[land_mask_np]))
r10_bias = r10_pred - r10_target

_p_cdd = (pred_mm_np_2d   < 1).sum(axis=0)
_t_cdd = (target_mm_np_2d < 1).sum(axis=0)
cdd_pred   = float(np.nanmean(_p_cdd[land_mask_np]))
cdd_target = float(np.nanmean(_t_cdd[land_mask_np]))
cdd_bias = cdd_pred - cdd_target

print(f'  {rx1d_label}_bias = {rx1d_bias:+.3f} mm (N={_n_test_days})  R10_bias = {r10_bias:+.3f}  CDD_bias = {cdd_bias:+.3f}')

# Climat expert I4 : drizzle_bias = fraction (pred wet > 1mm) - fraction (true wet > 1mm).
# Anti tail-weight over-correction monitor. Positive => model over-predicts wet days,
# negative => model under-predicts (under-correction). Computed in mm/day space
# (pred_mm_np_2d already in expm1 mm/day).
try:
    _wet = float(WET_DAY_THRESHOLD_MM)  # = 1.0
    _land_mask_3d = np.broadcast_to(land_mask_np[None, :, :], pred_mm_np_2d.shape)
    # M5 fix (IA Q5) : also filter NaN in pred -- otherwise pred NaN are silently
    # counted as 'dry' (False from comparison NaN >= 1.0), biasing drizzle_bias negative.
    _valid_3d = np.isfinite(target_mm_np_2d) & np.isfinite(pred_mm_np_2d) & _land_mask_3d
    if _valid_3d.sum() > 100:
        _pred_wet_frac = float((pred_mm_np_2d[_valid_3d] >= _wet).mean())
        _targ_wet_frac = float((target_mm_np_2d[_valid_3d] >= _wet).mean())
        drizzle_bias = _pred_wet_frac - _targ_wet_frac
        # M9 fix (Climat MEDIUM) : log filter_keep so user can detect if M5 isfinite(pred)
        # is removing a disproportionate amount of land pixels (e.g. NaN clustered on
        # mountain pixels would bias drizzle_bias positive).
        _mask_total = float(_land_mask_3d.sum())
        _filter_keep = float(_valid_3d.sum()) / max(_mask_total, 1.0)
        _targ_finite = _land_mask_3d & np.isfinite(target_mm_np_2d)
        _targ_wet_frac_unfilt = float((target_mm_np_2d[_targ_finite] >= _wet).mean()) if _targ_finite.sum() > 100 else float('nan')
        print(f'  drizzle_bias = {drizzle_bias:+.4f} (pred_wet_frac={_pred_wet_frac:.3f} vs targ_wet_frac={_targ_wet_frac:.3f})')
        print(f'  drizzle_filter_keep = {_filter_keep:.3f} (targ_wet_frac unfilt={_targ_wet_frac_unfilt:.3f}, delta={_targ_wet_frac-_targ_wet_frac_unfilt:+.4f})')
    else:
        drizzle_bias = float('nan')
        print(f'  drizzle_bias : skipped (mask too sparse)')
except Exception as _e:
    drizzle_bias = float('nan')
    print(f'  drizzle_bias : skipped ({type(_e).__name__}: {_e})')

phase8_metrics = {
    'F1_p99_pooled': f1_p99_pool,
    'F1_p95_pooled': f1_p95_pool,
    'F1_p99_etccdi': etccdi_p99['f1'],
    'F1_p95_etccdi': etccdi_p95['f1'],
    'CSI_p99': csi, 'SEDI_p99': sedi,
    'FSS_p99_n9': fss_9, 'FSS_p99_n25': fss_25, 'FSS_p99_n51': fss_51,
    'RMSE': rmse, 'MAE': mae,
    'Pearson_global': pearson_global, 'Pearson_per_sample': pearson_ps,
    f'{rx1d_label}_bias': rx1d_bias, 'R10_bias': r10_bias, 'CDD_bias': cdd_bias,
    'drizzle_bias': drizzle_bias if 'drizzle_bias' in dir() else float('nan'),
    'alpha_final': alpha_final,
    'best_ema_decay': best_decay,
    'n_batches_eval': int(test_pred.shape[0]),
    'k_samples': K_SAMPLES,
    'n_steps_diff': N_STEPS_DIFF,
}
print(f'[Cell 13] Phase 8 metrics computed')


In [ ]:
# === Cell 14 : Tests statistiques (paired permutation + bootstrap BCa + Holm-Bonferroni) ===

def paired_permutation_test(deltas, n_perm=PAIRED_PERMUTATION_N, seed=SEED):
    """Paired permutation test on per-batch deltas.
    H0 : mean(delta) = 0. Two-sided p-value."""
    deltas = np.asarray(deltas)
    obs = float(np.mean(deltas))
    rng = np.random.default_rng(seed)
    count = 0
    for _ in range(n_perm):
        signs = rng.choice([-1, 1], size=len(deltas))
        if abs(float(np.mean(signs * deltas))) >= abs(obs):
            count += 1
    return obs, (count + 1) / (n_perm + 1)

def bootstrap_bca_ci(values, n_resample=BOOTSTRAP_N_RESAMPLES, alpha=0.05, seed=SEED):
    """BCa bootstrap CI (Efron 1987)."""
    values = np.asarray(values)
    n = len(values)
    if n < 5:
        return float('nan'), float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    # Bootstrap resamples
    boots = np.array([np.mean(rng.choice(values, size=n, replace=True)) for _ in range(n_resample)])
    theta_hat = float(np.mean(values))
    # Bias correction z0
    p = float(np.mean(boots < theta_hat))
    p = min(max(p, 1e-6), 1 - 1e-6)
    from scipy.stats import norm
    z0 = norm.ppf(p)
    # Acceleration a via jackknife -- FIX BUG #5 (Math) Efron 1987 convention preserved
    # and degenerate guard added when sample is too small or homogeneous.
    jack = np.array([np.mean(np.delete(values, i)) for i in range(n)])
    jm = jack.mean()
    diffs = jm - jack
    num = (diffs ** 3).sum()
    den_squared = (diffs ** 2).sum()
    if den_squared < 1e-15 or n < 5:
        a = 0.0
    else:
        a = num / (6.0 * den_squared ** 1.5)
    # Adjusted alpha levels
    z_lo = norm.ppf(alpha / 2); z_hi = norm.ppf(1 - alpha / 2)
    alpha_lo = norm.cdf(z0 + (z0 + z_lo) / (1 - a * (z0 + z_lo)))
    alpha_hi = norm.cdf(z0 + (z0 + z_hi) / (1 - a * (z0 + z_hi)))
    return float(np.quantile(boots, alpha_lo)), float(np.quantile(boots, alpha_hi)), theta_hat

def holm_bonferroni(p_values, alpha=0.05):
    """Holm-Bonferroni correction. Returns rejected[] array."""
    p = np.asarray(p_values)
    idx = np.argsort(p)
    n = len(p)
    rejected = np.zeros(n, dtype=bool)
    for i, k in enumerate(idx):
        threshold = alpha / (n - i)
        if p[k] <= threshold:
            rejected[k] = True
        else:
            break
    return rejected

# Per-batch F1@p99 pooled (for bootstrap CI on Phase 8 alone)
per_batch_f1 = []
for i in range(pred_full_log.shape[0]):
    p_i = pred_full_log[i:i+1].flatten()
    t_i = target_full_log[i:i+1].flatten()
    v_i = torch.isfinite(p_i) & torch.isfinite(t_i)
    if v_i.sum() < 100:
        per_batch_f1.append(np.nan)
        continue
    pv = p_i[v_i]; tv = t_i[v_i]
    thr = float(torch.quantile(tv, 0.99))
    tp = ((pv > thr) & (tv > thr)).sum().item()
    fp = ((pv > thr) & (tv <= thr)).sum().item()
    fn = ((pv <= thr) & (tv > thr)).sum().item()
    if (tp + fp) == 0 or (tp + fn) == 0:
        per_batch_f1.append(0.0); continue
    prec = tp / (tp + fp); rec = tp / (tp + fn)
    per_batch_f1.append(2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0)

per_batch_f1 = [x for x in per_batch_f1 if np.isfinite(x)]
lo, hi, theta = bootstrap_bca_ci(per_batch_f1)
print(f'[Cell 14] Phase 8 F1@p99 (per-batch) : theta_hat={theta:.4f}  CI95%=[{lo:.4f}, {hi:.4f}]')

# For now, we have Phase 8 only. Comparison with noncausal v4 + V5 requires re-eval
# of those models in the same convention. We document and load existing JSONs.
ref_jsons = {}
for name, p in [
    ('noncausal_v4', DRIVE_ROOT / 'ckpt_noncausal' / 'final_validation_metrics.json'),
    ('V5_causal',    DRIVE_ROOT / 'ckpt_v2_corrdiff_normal' / 'final_validation_metrics.json'),
]:
    if p.exists():
        try: ref_jsons[name] = json.loads(p.read_text())
        except Exception as e: print(f'  Failed to load {name} : {e}')

print(f'[Cell 14] Loaded reference JSONs : {list(ref_jsons.keys())}')
print(f'[Cell 14] Note : these are in Convention B (zero+pooled), comparison via Section A2 below')

# Statistical tests summary (Phase 8 internal)
stat_tests = {
    'phase8_F1_p99_per_batch_BCa_CI95': {'lo': lo, 'hi': hi, 'theta_hat': theta, 'n': len(per_batch_f1)},
    'bootstrap_n_resamples': BOOTSTRAP_N_RESAMPLES,
    'paired_permutation_n': PAIRED_PERMUTATION_N,
    'note': 'Paired tests vs noncausal/V5 require re-eval of those models in this notebook\'s exact convention.',
}
print(f'[Cell 14] Stats tests computed (full paired vs noncausal pending re-eval)')


In [ ]:
# === Cell 15 : 3-way comparison + JSON publication-ready ===

print('=' * 100)
print('PHASE 8 vs noncausal v4 vs V5 -- comparison (apples-to-apples where possible)')
print('=' * 100)
print()
print(f'{"Metric":<28}{"Phase 8":>12}{"V5_causal":>14}{"noncausal v4":>16}{"Note":>20}')
print('-' * 100)

ref_v5 = ref_jsons.get('V5_causal', {})
ref_nc = ref_jsons.get('noncausal_v4', {})

def _get(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict) and k in d: d = d[k]
        else: return default
    return d if d is not None else default

rows = [
    ('F1@p99 (pooled)',      phase8_metrics['F1_p99_pooled'], _get(ref_v5, 'f1_extremes', 'p99'), _get(ref_nc, 'f1_extremes', 'p99'), 'log1p, ours vs JSON*'),
    ('F1@p95 (pooled)',      phase8_metrics['F1_p95_pooled'], _get(ref_v5, 'f1_extremes', 'p95'), _get(ref_nc, 'f1_extremes', 'p95'), 'log1p, ours vs JSON*'),
    ('F1@p99 (ETCCDI)',      phase8_metrics['F1_p99_etccdi'], None, None, 'per-pixel land only'),
    ('CSI@p99',              phase8_metrics['CSI_p99'],       None, None, 'ETCCDI threshold'),
    ('SEDI@p99',             phase8_metrics['SEDI_p99'],      None, None, 'ETCCDI threshold'),
    ('FSS@n=9',              phase8_metrics['FSS_p99_n9'],    None, None, 'reflect bord'),
    ('FSS@n=25',             phase8_metrics['FSS_p99_n25'],   None, None, ''),
    ('FSS@n=51',             phase8_metrics['FSS_p99_n51'],   None, None, ''),
    ('RMSE',                 phase8_metrics['RMSE'],          _get(ref_v5, 'rmse'), _get(ref_nc, 'rmse'), 'log1p space'),
    ('MAE',                  phase8_metrics['MAE'],           _get(ref_v5, 'mae'), _get(ref_nc, 'mae'), 'log1p space'),
    ('Pearson global',       phase8_metrics['Pearson_global'],_get(ref_v5, 'pearson_corr', 'global'), _get(ref_nc, 'pearson_corr', 'global'), ''),
    ('Pearson per-sample',   phase8_metrics['Pearson_per_sample'], _get(ref_v5, 'pearson_corr', 'per_sample_avg'), _get(ref_nc, 'pearson_corr', 'per_sample_avg'), ''),
    ('Rx1day_bias (mm)',     phase8_metrics['Rx1day_bias'],   None, None, 'mm/day, our N batches'),
    ('R10_bias (count)',     phase8_metrics['R10_bias'],      None, None, ''),
    ('CDD_bias (count)',     phase8_metrics['CDD_bias'],      None, None, ''),
]

def _fmt(x):
    if x is None: return '--'
    if isinstance(x, float): return f'{x:+.4f}' if abs(x) > 1 else f'{x:.4f}'
    return str(x)

for label, p8, v5, nc, note in rows:
    print(f'{label:<28}{_fmt(p8):>12}{_fmt(v5):>14}{_fmt(nc):>16}{note:>20}')

print('-' * 100)
print('*JSON references in Convention B (zero+pooled, NOT ETCCDI). For true apples-to-apples,')
print(' those models must be re-evaluated in this notebook\'s convention.')
print()

# Compute deltas Phase 8 vs noncausal where comparable
deltas_vs_nc = {}
for label, p8, _v5, nc, _note in rows:
    if nc is not None and isinstance(p8, (int, float)) and isinstance(nc, (int, float)):
        deltas_vs_nc[label] = p8 - nc

print('Phase 8 vs noncausal v4 (positive = Phase 8 better, except RMSE/MAE/biases where lower is better) :')
for k, v in deltas_vs_nc.items():
    print(f'  {k:<28} delta = {v:+.4f}')

# Final JSON
final_results = {
    'phase': 'phase8_from_scratch',
    'pre_registration': PRE_REG_RECORD,
    'smoke_mode': SMOKE_MODE,
    'phase8_metrics': phase8_metrics,
    'reference_jsons': ref_jsons,
    'deltas_vs_noncausal_v4': deltas_vs_nc,
    'stat_tests': stat_tests,
    'caveats': [
        'Reference JSONs (V5, noncausal) are in Convention B (zero+pooled). Phase 8 reports both pooled (apples-to-apples) and ETCCDI per-pixel.',
        'Climate indices (Rx1day, R10, CDD) computed on N_TEST_BATCHES batches only. For full 730-day indices, need a second eval pass.',
        'Paired permutation test vs noncausal/V5 requires re-eval of those models in this same convention -- TODO.',
        'alpha learned = {:.4f}, regularised to prevent collapse to 0.'.format(alpha_final),
        'Best EMA decay selected via val RMSE sweep : {}'.format(best_decay),
    ],
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}

FINAL_RESULTS.parent.mkdir(parents=True, exist_ok=True)
FINAL_RESULTS.write_text(json.dumps(final_results, indent=2, default=str), encoding='utf-8')
print()
print(f'Saved : {FINAL_RESULTS}')

try:
    from google.colab import files
    files.download(str(FINAL_RESULTS))
except Exception:
    pass

# Save training history too
TRAINING_HISTORY.write_text(json.dumps({
    'stage1_history': stage1_history,
    'stage2_history': stage2_history,
}, indent=2, default=str), encoding='utf-8')
print(f'Saved : {TRAINING_HISTORY}')

print()
print('=' * 100)
print('Phase 8 notebook complete.')
print('Next steps :')
print('  1. Validate SMOKE_MODE=True run produces reasonable numbers (~15 min compute)')
print('  2. Set SMOKE_MODE=False, restart kernel, full run (~45-55h on A100 Pro+)')
print('  3. For true apples-to-apples vs noncausal/V5 : re-eval those checkpoints with this')
print('     notebook\'s exact protocol (replace mu_total computation, keep same eval cells).')
print('=' * 100)
